# Beyond Concordance: Conformal Calibration of Predicted MCI-to-AD Conversion Timing

Code for the paper *Beyond Concordance: A Conformal Calibration Study of Predicted Alzheimer's Disease Conversion Timing* (Nix, Zhang, Lan, for the Alzheimer's Disease Neuroimaging Initiative).

This notebook reproduces every number, table, and figure in the paper. It trains four survival models (Cox proportional hazards, Weibull accelerated failure time, gradient-boosted survival analysis, and DeepSurv) to predict time from mild cognitive impairment (MCI) to Alzheimer's disease (AD), wraps each in conformal prediction intervals, and measures where each observed conversion falls inside its interval with the within-interval percentile $\pi$ (ideal mean 0.5). It then compares fixed-width split conformal intervals with two adaptive constructions, CSD and CiPOT.

**Data.** ADNI data cannot be redistributed. Download `ADNIMERGE.csv` from ADNI and place it at `data/ADNIMERGE.csv` (see `data/README.md`).

| Notebook section | Paper |
|---|---|
| 2. Cohort | Section 4.1, Table 1 |
| 3. Preprocessing | Sections 4.2 to 4.4 |
| 4. Survival models | Section 5.1 |
| 5. Discrimination | Section 5.2, Table 2 |
| 6. Fixed-width conformal intervals | Sections 3.1, 5.2 (timing agreement), 5.3 |
| 7. Landmark analysis | Section 5.4, Table 3 |
| 8. Censoring | Section 5.5 |
| 9. Adaptive intervals | Sections 3.3, 5.5, 5.6, Tables 4 and 5, Figures 4 and 5 |
| 10. Five-fold validation | Section 5.7.1, Table 6 |
| 11. Feature importance | Section 5.7.2 |
| 12. Paper numbers and figures | All reported values, Figures 2, 3, and 6 |

**Run time.** About 2 to 3 hours on CPU. Flags that change it:

| Flag | Section | Default | Effect |
|---|---|---|---|
| `RETRAIN` | 1.3 | `True` | `False` reloads saved model checkpoints instead of retraining |
| `RUN_MODEL_COMPARISON_BOOTSTRAP` | 5.2 | `False` | Pairwise model comparison, 100 resamples (about 7 hours) |
| `RUN_KFOLD_BOOTSTRAP` | 7.4 | `True` | Five-fold landmark analysis |
| `RUN_INTERVAL_SHAPE_KFOLD` | 10 | `True` | Five-fold comparison of the three interval constructions (slowest step) |

## 1. Setup

### 1.1 Install
Clones the repository and installs the requirements. Skip this cell when running from a local clone.

In [ ]:
REPO_URL = 'https://github.com/savannahnix/adni-conformal-timing.git'
!git clone {REPO_URL}
%cd adni-conformal-timing
!pip install -r requirements.txt -q

### 1.2 Paths
Set `DATA_PATH` to your copy of `ADNIMERGE.csv`.

In [ ]:
import os
from pathlib import Path

# ── Set your paths here ───────────────────────────────────────────────────────
# REPO_DIR: the root of this repository.
# DATA_PATH: your local copy of ADNIMERGE.csv (see data/README.md).
#   Kaggle example: Path('/kaggle/input/<your-private-dataset>/ADNIMERGE.csv')
REPO_DIR  = Path.cwd() if (Path.cwd() / 'pipeline').exists() else Path.cwd().parent
DATA_PATH = REPO_DIR / 'data' / 'ADNIMERGE.csv'
# ─────────────────────────────────────────────────────────────────────────────
os.chdir(REPO_DIR)
assert DATA_PATH.exists(), f"ADNIMERGE.csv not found at {DATA_PATH}. See data/README.md."
print(f'REPO_DIR  : {REPO_DIR}')
print(f'DATA_PATH : {DATA_PATH}')

### 1.3 Configuration
Libraries, global constants, the random seed (42, used everywhere), and readable feature labels for plots.

In [ ]:
%cd pipeline

In [ ]:
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
import torch
import optuna
from pycox.evaluation import EvalSurv
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)


# ── Output folders (inside pipeline/) ────────────────────────────────────────
BASELINE_DIR   = Path.cwd()
CHECKPOINT_DIR = BASELINE_DIR / 'checkpoints'
FIG_DIR        = BASELINE_DIR / 'figures'
OUT_DIR        = BASELINE_DIR / 'outputs'

for _d in (CHECKPOINT_DIR, FIG_DIR, OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ── Global constants ──────────────────────────────────────────────────────────
RANDOM_SEED = 42
N_FOLDS     = 5
HORIZONS    = [3, 5]
np.random.seed(RANDOM_SEED)

RETRAIN = True   # False: load saved checkpoints instead of retraining

# ── Feature groups ────────────────────────────────────────────────────────────
COG_FEATURES = [
    'MMSE', 'CDRSB', 'ADAS11', 'ADAS13', 'LDELTOTAL',
    'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting',
    'RAVLT_perc_forgetting', 'FAQ', 'MOCA', 'mPACCdigit', 'mPACCtrailsB',
    'EcogPtTotal', 'EcogSPTotal',
]
CSF_FEATURES  = ['ABETA', 'TAU', 'PTAU', 'FDG', 'AV45', 'FBB']
MRI_FEATURES  = ['Hippocampus', 'Entorhinal', 'Ventricles',
                  'WholeBrain', 'Fusiform', 'MidTemp', 'ICV']
DEMO_FEATURES = ['AGE', 'PTEDUCAT', 'APOE4']

MRI_HARMONIZE_COLS = ['Hippocampus', 'Entorhinal', 'Ventricles',
                       'Fusiform', 'MidTemp', 'WholeBrain']

# ── Human-readable feature label map ─────────────────────────────────────────
FEATURE_LABELS = {
    # Demographics
    'AGE':                   'Age at Baseline',
    'PTGENDER_num':          'Sex (Male=1)',
    'PTEDUCAT':              'Years of Education',
    'age_edu_interaction':   'Age × Education',
    'APOE4':                 'APOE ε4 Allele Count',
    # Cognitive tests
    'MMSE':                  'MMSE (Global Cognition)',
    'CDRSB':                 'CDR Sum of Boxes (Severity)',
    'ADAS11':                'ADAS-Cog 11 (Memory)',
    'ADAS13':                'ADAS-Cog 13 (Extended)',
    'LDELTOTAL':             'Logical Memory II (Delayed Recall)',
    'RAVLT_immediate':       'RAVLT Immediate Recall',
    'RAVLT_learning':        'RAVLT Learning Rate',
    'RAVLT_forgetting':      'RAVLT Forgetting Rate',
    'RAVLT_perc_forgetting': 'RAVLT % Forgetting',
    'FAQ':                   'FAQ (Functional Activities)',
    'MOCA':                  'MoCA (Global Cognition)',
    'mPACCdigit':            'mPACC Digit Span',
    'mPACCtrailsB':          'mPACC Trails B',
    'EcogPtTotal':           'ECog Patient-Reported Total',
    'EcogSPTotal':           'ECog Study Partner Total',
    # MRI volumetrics (ICV-normalized)
    'Hippocampus_ICV':       'Hippocampal Volume (ICV-adj)',
    'Entorhinal_ICV':        'Entorhinal Cortex Volume (ICV-adj)',
    'Ventricles_ICV':        'Ventricular Volume (ICV-adj)',
    'Fusiform_ICV':          'Fusiform Gyrus Volume (ICV-adj)',
    'MidTemp_ICV':           'Middle Temporal Gyrus Volume (ICV-adj)',
    'WholeBrain_ICV':        'Whole Brain Volume (ICV-adj)',
    'hippo_entorhinal_ratio':'Hippocampus / Entorhinal Ratio',
    'midtemp_fusiform_ratio':'MidTemporal / Fusiform Ratio',
    # CSF / PET biomarkers
    'ABETA':                 'CSF Amyloid-β (pg/mL)',
    'TAU':                   'CSF Total Tau (pg/mL)',
    'PTAU':                  'CSF Phospho-Tau (pg/mL)',
    'FDG':                   'FDG-PET (Glucose Metabolism)',
    'AV45':                  'AV45-PET (Amyloid Burden)',
    'FBB':                   'Florbetaben-PET (Amyloid Burden)',
    # APOE4 interaction terms
    'APOE4_x_amyloid':       'APOE ε4 × Amyloid Load',
    'APOE4_x_tau':           'APOE ε4 × Tau Burden',
    'APOE4_x_hippo_ICV':     'APOE ε4 × Hippocampal Volume',
    'APOE4_amyloid_pos':     'APOE ε4 × Amyloid Positivity',
    # Engineered / composite features
    'amyloid_composite':          'Amyloid Composite (CSF + PET)',
    'neurodegeneration_composite':'Neurodegeneration Composite (Tau + FDG)',
    'ATN_composite':              'ATN Biology Score',
    'amyloid_positive':           'Amyloid Positive (binary)',
    'cog_composite_pos':          'Cognitive Ability Composite',
    'severity_composite':         'Cognitive Severity Composite',
    'ecog_sp_pt_ratio':           'ECog Study-Partner / Patient Ratio',
    'n_modalities':               'Number of Available Modalities',
}

def feat_label(col):
    """Return the readable label for a feature column name."""
    return FEATURE_LABELS.get(col, col)

In [ ]:
import modeling
modeling.CHECKPOINT_DIR = CHECKPOINT_DIR
modeling.RANDOM_SEED = RANDOM_SEED
modeling.N_FOLDS = N_FOLDS
modeling.HORIZONS = HORIZONS
modeling.FIG_DIR = FIG_DIR
modeling.OUT_DIR = OUT_DIR

from modeling import save_checkpoint, load_checkpoint

import preprocessing
preprocessing.MRI_HARMONIZE_COLS = MRI_HARMONIZE_COLS
preprocessing.RANDOM_SEED = RANDOM_SEED
preprocessing.FIG_DIR = FIG_DIR

import postprocessing
postprocessing.FIG_DIR = FIG_DIR
postprocessing.OUT_DIR = OUT_DIR

## 2. Cohort (Section 4.1)

### 2.1 Load ADNIMERGE
Diagnosis labels from all ADNI phases are mapped to three states (CN, MCI, AD), and CSF values recorded as strings such as `>1700` are converted to numbers.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Raw shape: {df.shape}  |  Subjects: {df["RID"].nunique()}')

# ── Remap all diagnosis variants to 3 states ───────────────────────
dx_remap = {
    'CN': 'CN',   'SMC': 'CN',
    'MCI': 'MCI', 'EMCI': 'MCI', 'LMCI': 'MCI',
    'AD': 'AD', 'Dementia': 'AD', 'Dem': 'AD',
}
df['DX']    = df['DX'].map(dx_remap)
df['DX_bl'] = df['DX_bl'].map(dx_remap)

unmapped = df['DX'].isna().sum()
print(f'Rows with no recorded diagnosis (expected in ADNI): {unmapped}')

# ── Parse CSF strings that contain '>' or '<' ────────────────────────────────
for col in ['ABETA', 'TAU', 'PTAU', 'ABETA_bl', 'TAU_bl', 'PTAU_bl']:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace('>', '').str.replace('<', '').str.strip(),
        errors='coerce'
    )

# ── Coerce all numeric columns ────────────────────────────────────────────────
numeric_cols = (
    DEMO_FEATURES + COG_FEATURES + CSF_FEATURES + MRI_FEATURES +
    ['Years_bl', 'Month_bl', 'mPACCdigit', 'mPACCtrailsB']
)
for c_ in numeric_cols:
    if c_ in df.columns:
        df[c_] = pd.to_numeric(df[c_], errors='coerce')

df = df.sort_values(['RID', 'Years_bl']).reset_index(drop=True)

df_bl = df[df['VISCODE'] == 'bl'].copy()
print(f'\nBaseline subjects: {len(df_bl)}')
print(df_bl['DX_bl'].value_counts())

### 2.2 Transient reversions
Subjects who revert from MCI to cognitively normal are excluded as measurement noise, except those who later progress to AD.

In [ ]:
from preprocessing import classify_reverters

reversion_groups = classify_reverters(df)
for g, rids in reversion_groups.items():
    print(f'  {g:<22}: {len(rids)} subjects')

# Exclude transient reverters; keep subjects who later progress to AD
REVERSION_EXCLUSIONS = (reversion_groups['transient_noise'] |
                        reversion_groups['bouncer'])

print(f'\nExcluded: {len(REVERSION_EXCLUSIONS)} subjects')
print(f'Kept (later progressed to AD): {len(reversion_groups["progressor"])} subjects')

### 2.3 Survival labels
The event is the first observed AD diagnosis after a baseline MCI diagnosis; duration runs from baseline to conversion or to the last visit.

In [ ]:
from preprocessing import build_survival_labels

surv_mci = build_survival_labels(df, df_bl, 'MCI', 'AD', exclusion_rids=REVERSION_EXCLUSIONS)
print(f'MCI -> AD: {surv_mci["event"].sum()} events / {len(surv_mci)} subjects '
      f'({surv_mci["event"].mean()*100:.1f}%)')

## 3. Preprocessing (Sections 4.2 to 4.4)

### 3.1 Fix the split before preprocessing
The cohort is split 70/20/10 into training, calibration, and evaluation subjects, stratified on the event indicator. ComBat, the normalization constants, and the MICE imputer are then fit on the training subjects only and applied unchanged to the other two sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from neuroCombat import neuroCombat
from preprocessing import assemble_cohort

def _split_rids(surv_labels):
    slopes = pd.DataFrame({'RID': df_bl['RID'].unique()})
    _, ev, _, rids = assemble_cohort(df_bl, surv_labels, slopes, ['AGE'], [])
    idx = np.arange(len(rids))
    i_tr, i_tmp, _, ev_tmp = train_test_split(idx, ev, test_size=0.30,
                                              random_state=RANDOM_SEED, stratify=ev)
    i_cal, i_ev = train_test_split(i_tmp, test_size=0.333,
                                   random_state=RANDOM_SEED, stratify=ev_tmp)
    rids = np.asarray(rids)
    return {'train': set(rids[i_tr]), 'cal': set(rids[i_cal]), 'eval': set(rids[i_ev])}

MCI_SPLIT = _split_rids(surv_mci)
FIT_RIDS = set(MCI_SPLIT['train'])   # preprocessing is fit on these subjects only
print(f"MCI split: train={len(MCI_SPLIT['train'])}, cal={len(MCI_SPLIT['cal'])}, eval={len(MCI_SPLIT['eval'])}")

def mice_fit_transform(train_df, other_dfs, max_iter=10, seed=RANDOM_SEED):
    """MICE fit on the training rows only, then applied to the other sets."""
    imp = IterativeImputer(max_iter=max_iter, random_state=seed, keep_empty_features=True)
    tr = pd.DataFrame(imp.fit_transform(train_df), columns=train_df.columns, index=train_df.index)
    others = [pd.DataFrame(imp.transform(d), columns=d.columns, index=d.index) for d in other_dfs]
    return tr, others

def run_combat_train_only(df_baseline, fit_rids, cols, neuroCombat):
    """ComBat with batch = scanner field strength. Estimates come from `fit_rids` only and
    are applied to every subject. Missing values are median-filled (training medians) only
    inside the fit and stay missing in the output."""
    has = df_baseline[cols[0]].notna() & df_baseline['FLDSTRENG'].notna()
    for col in cols:
        df_baseline[f'{col}_raw'] = df_baseline[col].copy()
    batch = df_baseline['FLDSTRENG'].map({'1.5 Tesla MRI': 1, '3 Tesla MRI': 2})
    fit = has & df_baseline['RID'].isin(fit_rids)
    data_fit = df_baseline.loc[fit, cols].astype(float)
    data_fit = data_fit.fillna(data_fit.median())
    res = neuroCombat(dat=data_fit.T.values, covars=pd.DataFrame({'batch': batch[fit].values}), batch_col='batch')
    est = res['estimates']
    grand = np.asarray(est['stand.mean'])[:, 0]
    sd_pool = np.sqrt(np.asarray(est['var.pooled'])[:, 0])
    gamma, delta = np.asarray(est['gamma.star']), np.asarray(est['delta.star'])
    levels = np.sort(np.unique(batch[fit].values))          # neuroCombat recodes batches in sorted order
    X = df_baseline.loc[has, cols].values.astype(float)     # NaN stays NaN
    b = batch[has].values
    out = np.full_like(X, np.nan)
    for k, lvl in enumerate(levels):
        r = b == lvl
        out[r] = ((X[r] - grand) / sd_pool - gamma[k]) / np.sqrt(delta[k]) * sd_pool + grand
    df_baseline.loc[has, cols] = out
    print(f'  ComBat fit on {int(fit.sum())} subjects, applied to {int(has.sum())}')
    return df_baseline, res, fit

### 3.2 Scanner batch effect
Kruskal-Wallis test of ICV-normalized hippocampal volume across ADNI phases.

In [ ]:
df_bl['Hippocampus_ICV_raw'] = df_bl['Hippocampus'] / df_bl['ICV']
groups_kw = [
    grp['Hippocampus_ICV_raw'].dropna()
    for _, grp in df_bl.dropna(subset=['Hippocampus_ICV_raw']).groupby('COLPROT')
    if len(grp) > 5
]
if len(groups_kw) >= 2:
    stat, p = stats.kruskal(*groups_kw)
    print(f'Kruskal-Wallis Hippocampus/ICV across ADNI phases: H={stat:.2f}, p={p:.4f}')

### 3.3 ComBat harmonization
The six MRI volumes are harmonized with ComBat using field strength as the batch variable, then normalized by intracranial volume (ICV).

In [ ]:
from preprocessing import run_combat

# ── Infer missing FLDSTRENG from COLPROT ─────────────────────────────────────
protocol_to_field = {
    'ADNI1':  '1.5 Tesla MRI', 'ADNIGO': '1.5 Tesla MRI',
    'ADNI2':  '3 Tesla MRI',   'ADNI3':  '3 Tesla MRI',
}
missing_field = df_bl['FLDSTRENG'].isna()
df_bl.loc[missing_field, 'FLDSTRENG'] = df_bl.loc[missing_field, 'COLPROT'].map(protocol_to_field)
print(f"Inferred FLDSTRENG for {missing_field.sum()} subjects from COLPROT.")
print(df_bl['FLDSTRENG'].value_counts())

df_bl = df_bl.reset_index(drop=True)

print("\nRunning ComBat...")
df_bl, combat_result, combat_fit_mask = run_combat_train_only(df_bl, FIT_RIDS, MRI_HARMONIZE_COLS, neuroCombat)

df_bl['ICV'] = pd.to_numeric(df_bl['ICV'], errors='coerce')
for region in MRI_HARMONIZE_COLS:
    df_bl[f'{region}_ICV'] = df_bl[region] / df_bl['ICV']

print("Hippocampus_ICV by COLPROT:")
print(df_bl.groupby('COLPROT')['Hippocampus_ICV'].mean().round(6))

In [ ]:
from preprocessing import harmonization_report, plot_before_after


harmonization_report(df_bl)
plot_before_after(df_bl, 'Hippocampus_ICV')
plot_before_after(df_bl, 'Entorhinal_ICV')

### 3.4 Engineered features
ICV-normalized volumes and the four APOE4 interaction terms. Standardization constants come from the training subjects only.

In [ ]:
_FIT = df_bl['RID'].isin(FIT_RIDS)   # normalization constants from training subjects only

# ── ICV-normalized MRI volumes ───────────────────────────────────────────────
for region in ['Hippocampus', 'Entorhinal', 'Ventricles',
               'Fusiform', 'MidTemp', 'WholeBrain']:
    if f'{region}_ICV' not in df_bl.columns:
        df_bl[f'{region}_ICV'] = df_bl[region] / df_bl['ICV']

# ── Amyloid composite (ABETA CSF + AV45 PET combined) ────────────────────────
for feat, col in [('ABETA','ABETA_z'), ('AV45','AV45_z')]:
    mu, sd = df_bl.loc[_FIT, feat].mean(), df_bl.loc[_FIT, feat].std()
    df_bl[col] = (df_bl[feat] - mu) / (sd + 1e-9)
df_bl['ABETA_z_flipped'] = -df_bl['ABETA_z']
df_bl['amyloid_composite'] = df_bl[['ABETA_z_flipped','AV45_z']].mean(axis=1, skipna=True)
df_bl['amyloid_positive'] = (df_bl['ABETA'] < 880).astype(float)
df_bl.loc[df_bl['ABETA'].isna() & df_bl['AV45'].notna(), 'amyloid_positive'] = (
    df_bl.loc[df_bl['ABETA'].isna() & df_bl['AV45'].notna(), 'AV45'] > 1.11
).astype(float)

# ── Neurodegeneration composite (TAU + PTAU + FDG) ───────────────────────────
for feat, col in [('TAU','TAU_z'), ('PTAU','PTAU_z')]:
    mu, sd = df_bl.loc[_FIT, feat].mean(), df_bl.loc[_FIT, feat].std()
    df_bl[col] = (df_bl[feat] - mu) / (sd + 1e-9)
mu, sd = df_bl.loc[_FIT, 'FDG'].mean(), df_bl.loc[_FIT, 'FDG'].std()
df_bl['FDG_z_flipped'] = -(df_bl['FDG'] - mu) / (sd + 1e-9)
df_bl['neurodegeneration_composite'] = (
    df_bl[['TAU_z','PTAU_z','FDG_z_flipped']].mean(axis=1, skipna=True)
)

# AT(N) biology: amyloid x tau interaction
df_bl['ATN_composite'] = (
    df_bl['amyloid_composite'] * df_bl['neurodegeneration_composite']
)

# ── Cognitive composites ──────────────────────────────────────────────────────
for feat in ['MMSE', 'LDELTOTAL', 'RAVLT_immediate']:
    mu, sd = df_bl.loc[_FIT, feat].mean(), df_bl.loc[_FIT, feat].std()
    df_bl[f'{feat}_z'] = (df_bl[feat] - mu) / (sd + 1e-9)
df_bl['cog_composite_pos'] = (
    df_bl[['MMSE_z','LDELTOTAL_z','RAVLT_immediate_z']].mean(axis=1, skipna=True)
)

for feat, col in [('CDRSB','CDRSB_z_sev'),('FAQ','FAQ_z_sev'),('ADAS13','ADAS13_z_sev')]:
    mu, sd = df_bl.loc[_FIT, feat].mean(), df_bl.loc[_FIT, feat].std()
    df_bl[col] = (df_bl[feat] - mu) / (sd + 1e-9)
df_bl['severity_composite'] = (
    df_bl[['CDRSB_z_sev','FAQ_z_sev','ADAS13_z_sev']].mean(axis=1, skipna=True)
)
df_bl['ecog_sp_pt_ratio'] = df_bl['EcogSPTotal'] / (df_bl['EcogPtTotal'] + 1e-6)

# ── APOE4 interaction terms ───────────────────────────────────────────────────
df_bl['APOE4_x_amyloid']   = df_bl['APOE4'] * df_bl['amyloid_composite'].fillna(0)
df_bl['APOE4_x_tau']       = df_bl['APOE4'] * df_bl['TAU_z'].fillna(0)
df_bl['APOE4_x_hippo_ICV'] = df_bl['APOE4'] * df_bl['Hippocampus_ICV'].fillna(
    df_bl.loc[_FIT, 'Hippocampus_ICV'].median())
df_bl['APOE4_amyloid_pos'] = df_bl['APOE4'] * df_bl['amyloid_positive'].fillna(0)

# ── Multimodal completeness score ────────────────────────────────────────────
df_bl['n_modalities'] = (
    df_bl['MMSE'].notna().astype(int)
    + df_bl['ABETA'].notna().astype(int)
    + df_bl['AV45'].notna().astype(int)
    + df_bl['FDG'].notna().astype(int)
    + df_bl['Hippocampus'].notna().astype(int)
    + df_bl['APOE4'].notna().astype(int)
)

# ── Demographics ─────────────────────────────────────────────────────────────
df_bl['PTGENDER_num']        = (df_bl['PTGENDER'] == 'Male').astype(int)
df_bl['age_edu_interaction'] = df_bl['AGE'] * df_bl['PTEDUCAT']

# ── Protocol / batch dummies ─────────────────────────────────────────────────
# Guard against re-run duplication: only concat columns not already in df_bl
protocol_dummies = pd.get_dummies(df_bl['COLPROT'], prefix='prot', drop_first=True)
field_dummies    = pd.get_dummies(df_bl['FLDSTRENG'], prefix='field', drop_first=True)

new_prot  = [c for c in protocol_dummies.columns if c not in df_bl.columns]
new_field = [c for c in field_dummies.columns    if c not in df_bl.columns]

if new_prot:
    df_bl = pd.concat([df_bl, protocol_dummies[new_prot]], axis=1)
if new_field:
    df_bl = pd.concat([df_bl, field_dummies[new_field]], axis=1)

# Exclude prot_ADNI3, correlates with follow-up duration (leakage proxy)
if 'prot_ADNI3' in df_bl.columns:
    df_bl = df_bl.drop(columns=['prot_ADNI3'])

### 3.5 Cohort assembly and imputation
The 35 candidate features (Section 4.2), split into the training, calibration, and evaluation sets fixed in Section 3.1. MICE is fit on the training rows only.

In [ ]:
from preprocessing import mice_impute, assemble_cohort
from sklearn.model_selection import train_test_split

core_features = [
    # Demographics and genetics
    'AGE', 'PTGENDER_num', 'PTEDUCAT', 'APOE4',
    # Cognitive assessments
    'MMSE', 'CDRSB', 'ADAS11', 'ADAS13', 'LDELTOTAL',
    'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting',
    'FAQ', 'MOCA', 'EcogPtTotal', 'EcogSPTotal',
    'mPACCdigit', 'mPACCtrailsB',
    # MRI volumes, ICV-normalized, and two volume ratios
    'Hippocampus_ICV', 'Entorhinal_ICV', 'Ventricles_ICV',
    'Fusiform_ICV', 'MidTemp_ICV', 'WholeBrain_ICV',
    'hippo_entorhinal_ratio', 'midtemp_fusiform_ratio',
    # CSF and PET biomarkers
    'ABETA', 'TAU', 'PTAU', 'FDG', 'AV45',
    # APOE4 interaction terms
    'APOE4_x_amyloid', 'APOE4_x_tau', 'APOE4_x_hippo_ICV', 'APOE4_amyloid_pos',
]

no_slopes = pd.DataFrame({'RID': df_bl['RID'].unique()})   # baseline features only
X_mci_raw, y_ev_mci, y_dur_mci, rids_mci = assemble_cohort(df_bl, surv_mci, no_slopes, core_features, [])
X_mci_raw = X_mci_raw.loc[:, ~X_mci_raw.columns.duplicated()]
print(f'MCI cohort: {X_mci_raw.shape[0]} subjects x {X_mci_raw.shape[1]} features | '
      f'events={y_ev_mci.sum()} ({y_ev_mci.mean()*100:.1f}%)')

# 70 / 20 / 10 stratified split (the same subjects as Section 3.1)
X_mci_train, X_mci_tmp, y_ev_mci_train, y_ev_mci_tmp, y_dur_mci_train, y_dur_mci_tmp = (
    train_test_split(X_mci_raw, y_ev_mci, y_dur_mci,
                     test_size=0.30, random_state=RANDOM_SEED, stratify=y_ev_mci))
X_mci_cal, X_mci_eval, y_ev_mci_cal, y_ev_mci_eval, y_dur_mci_cal, y_dur_mci_eval = (
    train_test_split(X_mci_tmp, y_ev_mci_tmp, y_dur_mci_tmp,
                     test_size=0.333, random_state=RANDOM_SEED, stratify=y_ev_mci_tmp))

# MICE fit on the training rows only
X_mci_train, (X_mci_cal, X_mci_eval) = mice_fit_transform(X_mci_train, [X_mci_cal, X_mci_eval])
X_mci_imp = pd.concat([X_mci_train, X_mci_cal, X_mci_eval]).loc[X_mci_raw.index]

# Discrimination is reported on the calibration set; the model sections refer to it as "test"
X_mci_test, y_ev_mci_test, y_dur_mci_test = X_mci_cal, y_ev_mci_cal, y_dur_mci_cal

for name, X, ev in [('train', X_mci_train, y_ev_mci_train), ('calibration', X_mci_cal, y_ev_mci_cal),
                    ('evaluation', X_mci_eval, y_ev_mci_eval)]:
    print(f'{name:<12}: n={len(X)}, events={ev.sum()}')
feature_names_mci = list(X_mci_imp.columns)

### 3.6 Table 1: baseline characteristics
Raw ADNIMERGE values (before harmonization and imputation) for converters and non-converters. Prints the rows of Table 1.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency

base = df.copy()
if 'VISCODE' in base.columns and (base['VISCODE'] == 'bl').any():
    base = base[base['VISCODE'] == 'bl']
else:   # fall back to each subject's earliest visit
    sort_col = 'EXAMDATE' if 'EXAMDATE' in base.columns else base.columns[0]
    base = base.sort_values(['RID', sort_col])
base = base.drop_duplicates('RID').set_index('RID')

cohort = pd.DataFrame({'event': np.asarray(y_ev_mci, int)}, index=np.asarray(rids_mci))
tab = base.reindex(cohort.index)
tab['event'] = cohort['event']

def _num(s):  # ADNIMERGE stores some CSF values as strings like '>1700'
    return pd.to_numeric(s.astype(str).str.replace(r'[<>]', '', regex=True), errors='coerce')

if 'Hippocampus' in tab and 'ICV' in tab:
    tab['Hippo_ICV_x1000'] = _num(tab['Hippocampus']) / _num(tab['ICV']) * 1000

CONT = [('Age, years', 'AGE'), ('Education, years', 'PTEDUCAT'),
        ('MMSE', 'MMSE'), ('CDR-SB', 'CDRSB'), ('ADAS-Cog 13', 'ADAS13'), ('FAQ', 'FAQ'),
        ('Hippocampus/ICV ($\\times 10^{3}$)', 'Hippo_ICV_x1000'),
        ('CSF amyloid-beta, pg/mL', 'ABETA'), ('CSF total tau, pg/mL', 'TAU'),
        ('CSF p-tau, pg/mL', 'PTAU'), ('FDG-PET', 'FDG'), ('AV45-PET SUVR', 'AV45')]

conv, non = tab[tab.event == 1], tab[tab.event == 0]
print(f'n: all={len(tab)}, converters={len(conv)}, non-converters={len(non)}')
lines = [f'$n$ & {len(tab)} & {len(conv)} & {len(non)} & \\\\']

# Sex
if 'PTGENDER' in tab:
    def _female(d): return (d['PTGENDER'].astype(str).str.lower().str.startswith('f')).sum()
    ct = np.array([[_female(conv), len(conv) - _female(conv)], [_female(non), len(non) - _female(non)]])
    p = chi2_contingency(ct)[1]
    lines.append(f'Female, $n$ (\\%) & {_female(tab)} ({100*_female(tab)/len(tab):.1f}) & '
                 f'{_female(conv)} ({100*_female(conv)/len(conv):.1f}) & '
                 f'{_female(non)} ({100*_female(non)/len(non):.1f}) & {p:.3g} \\\\')
# APOE4 carriers
if 'APOE4' in tab:
    a = _num(tab['APOE4'])
    car = lambda idx: int((a.loc[idx] >= 1).sum())
    kn = lambda idx: int(a.loc[idx].notna().sum())
    ct = np.array([[car(conv.index), kn(conv.index) - car(conv.index)],
                   [car(non.index), kn(non.index) - car(non.index)]])
    p = chi2_contingency(ct)[1]
    lines.append(f'APOE4 carrier, $n$ (\\%) & {car(tab.index)} ({100*car(tab.index)/kn(tab.index):.1f}) & '
                 f'{car(conv.index)} ({100*car(conv.index)/kn(conv.index):.1f}) & '
                 f'{car(non.index)} ({100*car(non.index)/kn(non.index):.1f}) & {p:.3g} \\\\')

missing_notes = []
for label, col in CONT:
    if col not in tab:
        print(f'  (skipped {label}: column {col} not found)')
        continue
    x = _num(tab[col]); xc = x.loc[conv.index].dropna(); xn = x.loc[non.index].dropna()
    p = mannwhitneyu(xc, xn).pvalue if len(xc) and len(xn) else np.nan
    fmt = lambda v: f'{v.mean():.1f} ({v.std():.1f})' if abs(v.mean()) >= 10 else f'{v.mean():.2f} ({v.std():.2f})'
    p_str = '$<$0.001' if p < 0.001 else f'{p:.3f}'
    lines.append(f'{label} & {fmt(x.dropna())} & {fmt(xc)} & {fmt(xn)} & {p_str} \\\\')
    n_miss = int(x.isna().sum())
    if n_miss:
        missing_notes.append(f'{label}: {n_miss} missing ({100*n_miss/len(x):.0f}%)')

print('\nTable 1 rows:')
print('\n'.join(lines))
print('\nMissing at baseline (for the table note):')
print('; '.join(missing_notes))
(OUT_DIR / 'table1_baseline_rows.tex').write_text('\n'.join(lines) + '\n% Missing: ' + '; '.join(missing_notes) + '\n')

## 4. Survival models (Section 5.1)
Each model is tuned with Optuna (5-fold cross-validation within the training split) and evaluated with the IPCW time-dependent concordance (Antolini C-td) on the calibration set, with 500-resample bootstrap confidence intervals. Survival curves are also computed for the evaluation set, which is used for the interval analyses.

In [ ]:
results = {}   # model -> (C-td, CI lower, CI upper)

### 4.1 Cox proportional hazards
Elastic-net Cox model on a 29-feature subset (collinear interaction and ratio features removed).

In [ ]:
from modeling import run_cox_ph, calc_cox_ph_c, bootstrap_cindex_td, load_checkpoint, save_checkpoint

collinear_cols = [
    'age_edu_interaction', 'hippo_entorhinal_ratio', 'midtemp_fusiform_ratio',
    'APOE4_x_amyloid', 'APOE4_x_tau', 'APOE4_x_hippo_ICV', 'APOE4_amyloid_pos',
]
_drop = lambda X: X.drop(columns=[c for c in collinear_cols if c in X.columns])
X_mci_train_cox, X_mci_test_cox = _drop(X_mci_train), _drop(X_mci_test)
X_mci_cal_cox, X_mci_eval_cox = _drop(X_mci_cal), _drop(X_mci_eval)
print(f'Cox features: {X_mci_train_cox.shape[1]}')

if RETRAIN:
    oof_c_cox_mci, train_c_cox_mci, cox_model_mci, cox_scaler_mci, _ = run_cox_ph(
        X_mci_train_cox, y_ev_mci_train, y_dur_mci_train, label='MCI->AD', n_trials=30)
    print(f'CV C-td: {oof_c_cox_mci:.4f} | train C-td: {train_c_cox_mci:.4f}')
    save_checkpoint('cox_mci', (cox_model_mci, cox_scaler_mci))
else:
    cox_model_mci, cox_scaler_mci = load_checkpoint('cox_mci')

# Calibration set: C-td with bootstrap CI
c_td_cox_mci, surv_cox_mci = calc_cox_ph_c(
    cox_model_mci, cox_scaler_mci, X_mci_cal_cox, y_ev_mci_cal, y_dur_mci_cal)
ci_cox_mci = bootstrap_cindex_td(
    y_ev_mci_cal, y_dur_mci_cal, surv_cox_mci, n_bootstrap=500, seed=RANDOM_SEED)
results['Cox PH'] = (c_td_cox_mci, ci_cox_mci['lower'], ci_cox_mci['upper'])
print(f'Cox PH C-td: {c_td_cox_mci:.4f}  95% CI [{ci_cox_mci["lower"]:.4f}, {ci_cox_mci["upper"]:.4f}]')

# Evaluation set: survival curves for the interval analyses
_, surv_cox_mci_eval = calc_cox_ph_c(
    cox_model_mci, cox_scaler_mci, X_mci_eval_cox, y_ev_mci_eval, y_dur_mci_eval)

### 4.2 Gradient-boosted survival analysis (GBSA)

In [ ]:
from sklearn.ensemble._gb import BaseGradientBoosting

# Compatibility: newer scikit-learn requires `criterion`, which scikit-survival's
# GradientBoostingSurvivalAnalysis does not pass. Fill in scikit-learn's previous default.
if not getattr(BaseGradientBoosting.__init__, '_criterion_patched', False):
    _orig_base_init = BaseGradientBoosting.__init__

    def _patched_base_init(self, *args, **kwargs):
        kwargs.setdefault('criterion', 'friedman_mse')
        _orig_base_init(self, *args, **kwargs)

    _patched_base_init._criterion_patched = True
    BaseGradientBoosting.__init__ = _patched_base_init

In [ ]:
from modeling import gbsa_survival_cv, calc_gbsa_c, bootstrap_cindex_td

if RETRAIN:
    oof_c_gbsa_mci, train_c_gbsa_mci, imp_mci, gbsa_model_mci, gbsa_study_mci, _ = gbsa_survival_cv(
        X_mci_train, y_ev_mci_train, y_dur_mci_train,
        feature_names=list(X_mci_train.columns), label='MCI->AD', n_trials=40)
    print(f'CV C-td: {oof_c_gbsa_mci:.4f} | train C-td: {train_c_gbsa_mci:.4f}')
    save_checkpoint('gbsa_mci', gbsa_model_mci)
else:
    gbsa_model_mci = load_checkpoint('gbsa_mci')

c_td_gbsa_mci, surv_gbsa_mci = calc_gbsa_c(gbsa_model_mci, X_mci_test, y_ev_mci_test, y_dur_mci_test)
_, surv_gbsa_mci_eval = calc_gbsa_c(gbsa_model_mci, X_mci_eval, y_ev_mci_eval, y_dur_mci_eval)

ci_gbsa_mci = bootstrap_cindex_td(y_ev_mci_test, y_dur_mci_test, surv_gbsa_mci, n_bootstrap=500)
results['GBSA'] = (c_td_gbsa_mci, ci_gbsa_mci['lower'], ci_gbsa_mci['upper'])
print(f'GBSA C-td: {c_td_gbsa_mci:.4f}  95% CI [{ci_gbsa_mci["lower"]:.4f}, {ci_gbsa_mci["upper"]:.4f}]')

### 4.3 Weibull accelerated failure time (AFT)
The penalizer is chosen by 5-fold cross-validated concordance.

In [ ]:
from lifelines import WeibullAFTFitter
from concordance import concordance_td
from modeling import bootstrap_cindex_td
from sklearn.model_selection import StratifiedKFold as _SKF

def _fit_aft_with_cv(X_tr, y_ev_tr, y_dur_tr, lbl):
    _skf = _SKF(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    _yev  = y_ev_tr.values  if hasattr(y_ev_tr,  'values') else np.array(y_ev_tr)
    _ydur = y_dur_tr.values if hasattr(y_dur_tr, 'values') else np.array(y_dur_tr)
    best_pen, best_c = 0.0, -1.0
    for pen in [0.0, 0.01, 0.1, 1.0, 5.0]:
        _fcs = []
        for tr, va in _skf.split(X_tr, _yev):
            _dtr = X_tr.iloc[tr].copy()
            _dtr['duration'] = _ydur[tr]; _dtr['event'] = _yev[tr]
            _dva = X_tr.iloc[va].copy()
            _dva['duration'] = _ydur[va]; _dva['event'] = _yev[va]
            try:
                _m = WeibullAFTFitter(penalizer=pen)
                _m.fit(_dtr, duration_col='duration', event_col='event')
                _fcs.append(_m.score(_dva, scoring_method='concordance_index'))
            except Exception: pass
        if _fcs:
            _mean = np.mean(_fcs)
            print(f'  [{lbl}] pen={pen:.2f}: CV C={_mean:.4f}')
            if _mean > best_c: best_c, best_pen = _mean, pen
    print(f'  [{lbl}] Best penalizer: {best_pen}  (CV C={best_c:.4f})')
    _data = X_tr.copy()
    _data['duration'] = _ydur; _data['event'] = _yev
    _model = WeibullAFTFitter(penalizer=best_pen)
    _model.fit(_data, duration_col='duration', event_col='event')
    return _model

if RETRAIN:
    aft_model_mci = _fit_aft_with_cv(X_mci_train, y_ev_mci_train, y_dur_mci_train, 'MCI->AD')
    save_checkpoint('aft_mci', aft_model_mci)
else:
    aft_model_mci = load_checkpoint('aft_mci')

# Calibration set: C-td with bootstrap CI
_dur_mci = y_dur_mci_cal.values if hasattr(y_dur_mci_cal, 'values') else np.array(y_dur_mci_cal)
_ev_mci  = np.array(y_ev_mci_cal, dtype=np.int32)
_test_mci = X_mci_cal.copy()
_test_mci['duration'] = _dur_mci; _test_mci['event'] = _ev_mci
aft_surv_mci = aft_model_mci.predict_survival_function(_test_mci)
_tg_mci = aft_surv_mci.index.values.astype(np.float64)
_idx_mci = np.clip(np.searchsorted(_tg_mci, _dur_mci), 0, len(_tg_mci)-1).astype(np.int64)
aft_c_mci = concordance_td(_dur_mci, _ev_mci, aft_surv_mci.values, _idx_mci,
                           method='adj_antolini', ipcw=True)
ci_aft_mci = bootstrap_cindex_td(_ev_mci, _dur_mci, aft_surv_mci, n_bootstrap=500)
results['AFT'] = (aft_c_mci, ci_aft_mci['lower'], ci_aft_mci['upper'])
print(f'AFT C-td: {aft_c_mci:.4f}  95% CI [{ci_aft_mci["lower"]:.4f}, {ci_aft_mci["upper"]:.4f}]')

### 4.4 DeepSurv

In [ ]:
from modeling import run_deepsurv, calc_deepsurv_c, bootstrap_cindex_td

if RETRAIN:
    oof_c_ds_mci, train_c_ds_mci, ds_model_mci, ds_scaler_mci, ds_loss_mci, ds_study_mci, _ = run_deepsurv(
        X_mci_train, y_ev_mci_train, y_dur_mci_train, 'MCI->AD', n_trials=25)
    print(f'CV C-td: {oof_c_ds_mci:.4f} | train C-td: {train_c_ds_mci:.4f}')
    ds_model_mci.callbacks = []
    save_checkpoint('deepsurv_mci_metadata',
                    (oof_c_ds_mci, train_c_ds_mci, ds_model_mci, ds_scaler_mci, ds_loss_mci))
else:
    oof_c_ds_mci, train_c_ds_mci, ds_model_mci, ds_scaler_mci, ds_loss_mci = load_checkpoint('deepsurv_mci_metadata')
    _X_tr = ds_scaler_mci.transform(X_mci_train.values).astype('float32')
    ds_model_mci.compute_baseline_hazards(
        input=_X_tr,
        target=(np.asarray(y_dur_mci_train, dtype='float32'), np.asarray(y_ev_mci_train, dtype='float32')),
        batch_size=256)

# Calibration set: C-td with bootstrap CI (surv_mci holds DeepSurv's calibration-set curves)
ds_c_mci, surv_mci = calc_deepsurv_c(ds_model_mci, ds_scaler_mci, X_mci_test, y_ev_mci_test, y_dur_mci_test)
ci_ds_mci = bootstrap_cindex_td(y_ev_mci_test, y_dur_mci_test, surv_mci, n_bootstrap=500)
results['DeepSurv'] = (ds_c_mci, ci_ds_mci['lower'], ci_ds_mci['upper'])
print(f'DeepSurv C-td: {ds_c_mci:.4f}  95% CI [{ci_ds_mci["lower"]:.4f}, {ci_ds_mci["upper"]:.4f}]')

# Evaluation set: survival curves for the interval analyses
_, surv_mci_eval = calc_deepsurv_c(ds_model_mci, ds_scaler_mci, X_mci_eval, y_ev_mci_eval, y_dur_mci_eval)

## 5. Discrimination (Section 5.2)

### 5.1 Table 2: time-dependent concordance
IPCW Antolini C-td on the calibration set (192 subjects), with 500-resample bootstrap 95% confidence intervals.

In [ ]:
results_df = pd.DataFrame(
    [dict(Model=m, **{'C-td': round(c, 4), 'CI lower': round(lo, 4), 'CI upper': round(hi, 4)})
     for m, (c, lo, hi) in results.items()]
).sort_values('C-td', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))
results_df.to_csv(OUT_DIR / 'model_comparison.csv', index=False)

### 5.2 Pairwise model comparison (optional, about 7 hours)
Each of 100 iterations resamples the training set with replacement, refits all four models with a short hyperparameter search, and scores them on the fixed evaluation set. Pivotal 95% intervals for each pairwise C-td difference are saved to `outputs/model_comparison_pairwise_ci.csv`. Results are written after every iteration, so an interrupted run resumes where it stopped.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd

RUN_MODEL_COMPARISON_BOOTSTRAP = False   # True to run (about 7 hours)
N_BOOTSTRAP = 100
BOOT_N_TRIALS = 8                        # short hyperparameter search per refit
CHECKPOINT_PATH = OUT_DIR / 'model_comparison_bootstrap.csv'

def _fit_and_score_cox(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, n_trials):
    _, _, model, scaler, _ = run_cox_ph(X_tr, ev_tr, dur_tr, label='BootCox', n_trials=n_trials)
    c_td, _ = calc_cox_ph_c(model, scaler, X_ev, ev_ev, dur_ev)
    return c_td

def _fit_and_score_gbsa(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, n_trials):
    _, _, _, model, _, _ = gbsa_survival_cv(X_tr, ev_tr, dur_tr,
        feature_names=list(X_tr.columns), label='BootGBSA', n_trials=n_trials)
    c_td, _ = calc_gbsa_c(model, X_ev, ev_ev, dur_ev)
    return c_td

def _fit_and_score_aft(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, n_trials=None):
    model = _fit_aft_with_cv(X_tr, ev_tr, dur_tr, 'BootAFT')
    _data = X_ev.copy()
    _data['duration'] = np.array(dur_ev, dtype=np.float64)
    _data['event'] = np.array(ev_ev, dtype=np.int32)
    surv = model.predict_survival_function(_data)
    tg = surv.index.values.astype(np.float64)
    idx = np.clip(np.searchsorted(tg, np.array(dur_ev)), 0, len(tg) - 1).astype(np.int64)
    return concordance_td(np.array(dur_ev, dtype=np.float64), np.array(ev_ev, dtype=np.int32),
                           surv.values, idx, method='adj_antolini', ipcw=True)

def _fit_and_score_deepsurv(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, n_trials):
    _, _, model, scaler, _, _, _ = run_deepsurv(X_tr, ev_tr, dur_tr, 'BootDeepSurv', n_trials=n_trials)
    c_td, _ = calc_deepsurv_c(model, scaler, X_ev, ev_ev, dur_ev)
    return c_td

FITTERS = {
    'Cox PH': _fit_and_score_cox,
    'GBSA': _fit_and_score_gbsa,
    'AFT': _fit_and_score_aft,
    'DeepSurv': _fit_and_score_deepsurv,
}

def _run_one_iteration(b):
    """One bootstrap iteration: resample the training set, refit all four models,
    and score them on the fixed evaluation set (never resampled)."""
    row = {'iteration': b}
    try:
        rng = np.random.default_rng(RANDOM_SEED + b)
        boot_idx = rng.choice(len(X_mci_train), size=len(X_mci_train), replace=True)
        X_tr = X_mci_train.iloc[boot_idx].reset_index(drop=True)
        ev_tr = (y_ev_mci_train.iloc[boot_idx].reset_index(drop=True)
                 if hasattr(y_ev_mci_train, 'iloc') else y_ev_mci_train[boot_idx])
        dur_tr = (y_dur_mci_train.iloc[boot_idx].reset_index(drop=True)
                  if hasattr(y_dur_mci_train, 'iloc') else y_dur_mci_train[boot_idx])

        n_events = int(np.sum(np.asarray(ev_tr) == 1))
        n_censored = int(np.sum(np.asarray(ev_tr) == 0))
        if n_events < 2 or n_censored < 2:
            row['error'] = f'degenerate resample (events={n_events}, censored={n_censored})'
            for name in FITTERS:
                row[name] = np.nan
            return row

        X_ev, ev_ev, dur_ev = X_mci_eval, y_ev_mci_eval, y_dur_mci_eval

        for name, fitter in FITTERS.items():
            try:
                row[name] = fitter(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, BOOT_N_TRIALS)
            except Exception as e:
                print(f'  [bootstrap {b}] {name} failed: {e}')
                row[name] = np.nan
        row['error'] = ''
    except Exception as e:
        print(f'  [bootstrap {b}] ITERATION-LEVEL failure (resampling/split step): {e}')
        row['error'] = str(e)
        for name in FITTERS:
            row.setdefault(name, np.nan)
    return row

if RUN_MODEL_COMPARISON_BOOTSTRAP:
    # Resume from any iterations already saved
    if CHECKPOINT_PATH.exists():
        done_df = pd.read_csv(CHECKPOINT_PATH)
        done_iters = set(done_df['iteration'].tolist())
        print(f'Found existing checkpoint with {len(done_iters)} completed iterations. Resuming.')
    else:
        done_df = pd.DataFrame(columns=['iteration'] + list(FITTERS.keys()) + ['error'])
        done_iters = set()
        print('No existing checkpoint found. Starting fresh.')

    start_time = time.time()
    for b in range(N_BOOTSTRAP):
        if b in done_iters:
            continue

        iter_start = time.time()
        row = _run_one_iteration(b)
        iter_elapsed = time.time() - iter_start

        row_df = pd.DataFrame([row])
        row_df.to_csv(CHECKPOINT_PATH, mode='a', header=not CHECKPOINT_PATH.exists(), index=False)
        done_df = pd.concat([done_df, row_df], ignore_index=True)

        n_done = len(done_df)
        elapsed_total = time.time() - start_time
        avg_per_iter = elapsed_total / max(1, (n_done - len(done_iters)))
        remaining = N_BOOTSTRAP - n_done
        eta_hrs = (remaining * avg_per_iter) / 3600

        scores_str = {k: (round(row[k], 4) if not pd.isna(row[k]) else 'NaN') for k in FITTERS}
        print(f'[{n_done}/{N_BOOTSTRAP}] iter {b} done in {iter_elapsed/60:.1f} min '
              f'| scores={scores_str} | ETA remaining: {eta_hrs:.1f} hr')

    print(f'\nAll {N_BOOTSTRAP} iterations complete (or already were). '
          f'Checkpoint at: {CHECKPOINT_PATH}')

    # ── Pivotal intervals on pairwise differences ────────────────────────────
    boot_df = pd.read_csv(CHECKPOINT_PATH)
    n_failed = boot_df['error'].fillna('').astype(bool).sum() if 'error' in boot_df else 0
    if n_failed:
        print(f'Note: {n_failed}/{len(boot_df)} iterations had at least one issue logged '
              f'check the "error" column before trusting the intervals below.')

    from itertools import combinations
    print('\nPairwise pivotal 95% CIs on C-td difference (model A - model B):')
    print('(interval containing 0 => not statistically distinguishable)\n')
    pairwise_results = []
    for m1, m2 in combinations(FITTERS.keys(), 2):
        diffs = (boot_df[m1] - boot_df[m2]).dropna().values
        if len(diffs) < 2:
            print(f'{m1} - {m2}: insufficient completed iterations to compute an interval.')
            continue
        point_diff = diffs.mean()
        lo = 2 * point_diff - np.percentile(diffs, 97.5)
        hi = 2 * point_diff - np.percentile(diffs, 2.5)
        contains_zero = lo <= 0 <= hi
        pairwise_results.append(dict(pair=f'{m1} - {m2}', mean_diff=point_diff,
                                      ci_lo=lo, ci_hi=hi, contains_zero=contains_zero,
                                      n=len(diffs)))
        print(f'{m1:>10} - {m2:<10}: mean diff={point_diff:+.4f}  95% CI=[{lo:+.4f}, {hi:+.4f}]  '
              f'n={len(diffs)}  {"NOT distinguishable" if contains_zero else "DISTINGUISHABLE"}')

    pd.DataFrame(pairwise_results).to_csv(OUT_DIR / 'model_comparison_pairwise_ci.csv', index=False)
else:
    print('Skipped (RUN_MODEL_COMPARISON_BOOTSTRAP = False).')

## 6. Fixed-width conformal intervals (Sections 3.1 and 5.3)
For each model, the predicted median conversion time $\hat T_i$ is the first time the survival curve reaches 0.5. Conformity scores $|T_i - \hat T_i|$ are computed on the calibration subjects with an observed conversion, and the interval is $[\hat T_i - \hat q, \hat T_i + \hat q]$, where $\hat q$ is the conformal quantile at level $1-\alpha$. Coverage and the within-interval percentile $\pi_i = (T_i - L_i)/(U_i - L_i)$ are measured on the evaluation subjects with an observed conversion. Results are reported at $\alpha = 0.10$ and $0.20$.

### 6.1 AFT

In [ ]:
ev_cal_aft   = np.array(y_ev_mci_cal,   dtype=np.int32)
ev_eval_aft  = np.array(y_ev_mci_eval,  dtype=np.int32)
dur_cal_aft  = np.array(y_dur_mci_cal,  dtype=np.float64)
dur_eval_aft = np.array(y_dur_mci_eval, dtype=np.float64)
X_cal_aft    = X_mci_cal.reset_index(drop=True)
X_eval_aft   = X_mci_eval.reset_index(drop=True)
print(f'AFT CP cal  : n={len(X_cal_aft)}, events={ev_cal_aft.sum()} ({ev_cal_aft.mean()*100:.1f}%)')
print(f'AFT CP eval : n={len(X_eval_aft)}, events={ev_eval_aft.sum()} ({ev_eval_aft.mean()*100:.1f}%)')

# ── Extract predicted median (in years) ───────────────────────────────────────
def _aft_predict_log_median(model, X):
    X_clean = X.drop(columns=['duration', 'event'], errors='ignore')
    median_t = model.predict_median(X_clean)
    return np.log(np.clip(median_t.values, 1e-6, None))

def _aft_sigma(model):
    rho_intercept = model.params_['rho_']['Intercept']
    return float(np.exp(-rho_intercept))

mu_cal  = _aft_predict_log_median(aft_model_mci, X_cal_aft)
mu_eval = _aft_predict_log_median(aft_model_mci, X_eval_aft)
sigma   = _aft_sigma(aft_model_mci)
print(f'Weibull scale sigma: {sigma:.4f}')
pred_median_cal  = np.exp(mu_cal)
pred_median_eval = np.exp(mu_eval)
print(f'Predicted median cal- mean: {pred_median_cal.mean():.3f} yr, '
      f'std: {pred_median_cal.std():.3f} yr')
print(f'Predicted median eval- mean: {pred_median_eval.mean():.3f} yr, '
      f'std: {pred_median_eval.std():.3f} yr')

# ── Nonconformity scores (observed events only, raw time scale) ───────────────
obs_cal_mask   = ev_cal_aft == 1
scores_cal_aft = pred_median_cal[obs_cal_mask] - dur_cal_aft[obs_cal_mask]
print(f'\nCalibration scores (observed only): n={len(scores_cal_aft)}, '
      f'mean={scores_cal_aft.mean():.3f}, std={scores_cal_aft.std():.3f}')

# ── Conformal quantile helper ─────────────────────────────────────────────────
def _conformal_quantile(scores, alpha, n_cal):
    level = min(np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, 1.0)
    return float(np.quantile(scores, level))

# ── Prediction intervals and coverage ────────────────────────────────────────
results_cp_aft = {}
n_cal_aft = len(scores_cal_aft)

for alpha in [0.10, 0.20]:
    q_hat   = _conformal_quantile(scores_cal_aft, alpha, n_cal_aft)
    lo      = np.clip(pred_median_eval - q_hat, 0, None)
    hi      = pred_median_eval + q_hat
    widths  = hi - lo
    obs     = ev_eval_aft == 1
    n_obs   = obs.sum()
    covered = ((dur_eval_aft[obs] >= lo[obs]) & (dur_eval_aft[obs] <= hi[obs])).sum()
    coverage = covered / n_obs if n_obs > 0 else np.nan
    print(f'alpha={alpha:.2f}  q_hat={q_hat:.4f} yr  '
          f'coverage={coverage:.1%} ({covered}/{n_obs})  '
          f'median_width={np.median(widths):.3f} yr')
    results_cp_aft[alpha] = dict(
        q_hat=q_hat, lo=lo, hi=hi, widths=widths,
        coverage=coverage, n_obs=n_obs, covered=covered,
        mu_eval=mu_eval, pred_median_eval=pred_median_eval,
    )

### 6.2 Cox PH

In [ ]:
ev_cal_cox    = np.array(y_ev_mci_cal,  dtype=np.int32)
ev_eval_cox   = np.array(y_ev_mci_eval, dtype=np.int32)
dur_cal_cox   = np.array(y_dur_mci_cal,  dtype=np.float64)
dur_eval_cox  = np.array(y_dur_mci_eval, dtype=np.float64)
surv_cox_cal  = surv_cox_mci
surv_cox_eval = surv_cox_mci_eval

print(f'Cox CP cal  : n={surv_cox_cal.shape[1]}, events={ev_cal_cox.sum()}')
print(f'Cox CP eval : n={surv_cox_eval.shape[1]}, events={ev_eval_cox.sum()}')

# ── Invert survival function at S(t) = 0.5 ───────────────────────────────────
def _invert_surv(surv_matrix, p=0.5):
    times   = surv_matrix.index.values.astype(np.float64)
    medians = np.full(surv_matrix.shape[1], times[-1])
    for j in range(surv_matrix.shape[1]):
        below = np.where(surv_matrix.iloc[:, j].values <= p)[0]
        if len(below) > 0:
            medians[j] = times[below[0]]
    return medians

t_hat_cal  = _invert_surv(surv_cox_cal)
t_hat_eval = _invert_surv(surv_cox_eval)
print(f'Cox predicted median: cal mean : {t_hat_cal.mean():.3f} yr')
print(f'Cox predicted median: eval mean: {t_hat_eval.mean():.3f} yr')

# ── Nonconformity scores (observed events only, raw time scale) ───────────────
obs_cal_mask_cox = ev_cal_cox == 1
scores_cal_cox = t_hat_cal[obs_cal_mask_cox] - dur_cal_cox[obs_cal_mask_cox]
print(f'Calibration scores (observed only): n={len(scores_cal_cox)}, '
      f'mean={scores_cal_cox.mean():.3f}, std={scores_cal_cox.std():.3f}')

# ── Conformal quantile helper (shared across all four models) ────────────────
def _conformal_quantile(scores, alpha, n_cal):
    level = min(np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, 1.0)
    return float(np.quantile(scores, level))

# ── Prediction intervals and coverage ────────────────────────────────────────
results_cp_cox = {}
n_cal_cox = len(scores_cal_cox)

for alpha in [0.10, 0.20]:
    q_hat   = _conformal_quantile(scores_cal_cox, alpha, n_cal_cox)
    lo      = np.clip(t_hat_eval - q_hat, 0, None)
    hi      = t_hat_eval + q_hat
    widths  = hi - lo
    obs     = ev_eval_cox == 1
    n_obs   = obs.sum()
    covered = ((dur_eval_cox[obs] >= lo[obs]) & (dur_eval_cox[obs] <= hi[obs])).sum()
    coverage = covered / n_obs if n_obs > 0 else np.nan
    print(f'alpha={alpha:.2f}  q_hat={q_hat:.4f} yr  '
          f'coverage={coverage:.1%} ({covered}/{n_obs})  '
          f'width={widths[0]:.3f} yr (constant)')
    results_cp_cox[alpha] = dict(q_hat=q_hat, lo=lo, hi=hi, widths=widths,
                                  coverage=coverage, n_obs=n_obs, covered=covered,
                                  t_hat_eval=t_hat_eval)

### 6.3 GBSA

In [ ]:
ev_cal_gbsa    = np.array(y_ev_mci_cal,  dtype=np.int32)
ev_eval_gbsa   = np.array(y_ev_mci_eval, dtype=np.int32)
dur_cal_gbsa   = np.array(y_dur_mci_cal,  dtype=np.float64)
dur_eval_gbsa  = np.array(y_dur_mci_eval, dtype=np.float64)
surv_gbsa_cal_ = surv_gbsa_mci
surv_gbsa_eval_ = surv_gbsa_mci_eval

print(f'GBSA CP cal  : n={surv_gbsa_cal_.shape[1]}, events={ev_cal_gbsa.sum()}')
print(f'GBSA CP eval : n={surv_gbsa_eval_.shape[1]}, events={ev_eval_gbsa.sum()}')

t_hat_cal_gbsa  = _invert_surv(surv_gbsa_cal_)
t_hat_eval_gbsa = _invert_surv(surv_gbsa_eval_)
print(f'GBSA predicted median: cal mean : {t_hat_cal_gbsa.mean():.3f} yr')
print(f'GBSA predicted median: eval mean: {t_hat_eval_gbsa.mean():.3f} yr')

obs_cal_mask_gbsa = ev_cal_gbsa == 1
scores_cal_gbsa = t_hat_cal_gbsa[obs_cal_mask_gbsa] - dur_cal_gbsa[obs_cal_mask_gbsa]
print(f'Calibration scores (observed only): n={len(scores_cal_gbsa)}, '
      f'mean={scores_cal_gbsa.mean():.3f}, std={scores_cal_gbsa.std():.3f}')

results_cp_gbsa = {}
n_cal_gbsa = len(scores_cal_gbsa)

for alpha in [0.10, 0.20]:
    q_hat   = _conformal_quantile(scores_cal_gbsa, alpha, n_cal_gbsa)
    lo      = np.clip(t_hat_eval_gbsa - q_hat, 0, None)
    hi      = t_hat_eval_gbsa + q_hat
    widths  = hi - lo
    obs     = ev_eval_gbsa == 1
    n_obs   = obs.sum()
    covered = ((dur_eval_gbsa[obs] >= lo[obs]) & (dur_eval_gbsa[obs] <= hi[obs])).sum()
    coverage = covered / n_obs if n_obs > 0 else np.nan
    print(f'alpha={alpha:.2f}  q_hat={q_hat:.4f} yr  '
          f'coverage={coverage:.1%} ({covered}/{n_obs})  '
          f'width={widths[0]:.3f} yr (constant)')
    results_cp_gbsa[alpha] = dict(q_hat=q_hat, lo=lo, hi=hi, widths=widths,
                                   coverage=coverage, n_obs=n_obs, covered=covered,
                                   t_hat_eval=t_hat_eval_gbsa)

### 6.4 DeepSurv

In [ ]:
ev_cal_ds    = np.array(y_ev_mci_cal,  dtype=np.int32)
ev_eval_ds   = np.array(y_ev_mci_eval, dtype=np.int32)
dur_cal_ds   = np.array(y_dur_mci_cal,  dtype=np.float64)
dur_eval_ds  = np.array(y_dur_mci_eval, dtype=np.float64)
surv_ds_cal_ = surv_mci
surv_ds_eval_ = surv_mci_eval

print(f'DeepSurv CP cal  : n={surv_ds_cal_.shape[1]}, events={ev_cal_ds.sum()}')
print(f'DeepSurv CP eval : n={surv_ds_eval_.shape[1]}, events={ev_eval_ds.sum()}')

t_hat_cal_ds  = _invert_surv(surv_ds_cal_)
t_hat_eval_ds = _invert_surv(surv_ds_eval_)
print(f'DeepSurv predicted median- cal mean : {t_hat_cal_ds.mean():.3f} yr')
print(f'DeepSurv predicted median- eval mean: {t_hat_eval_ds.mean():.3f} yr')

obs_cal_mask_ds = ev_cal_ds == 1
scores_cal_ds = t_hat_cal_ds[obs_cal_mask_ds] - dur_cal_ds[obs_cal_mask_ds]
print(f'Calibration scores (observed only): n={len(scores_cal_ds)}, '
      f'mean={scores_cal_ds.mean():.3f}, std={scores_cal_ds.std():.3f}')

results_cp_deepsurv = {}
n_cal_ds = len(scores_cal_ds)

for alpha in [0.10, 0.20]:
    q_hat   = _conformal_quantile(scores_cal_ds, alpha, n_cal_ds)
    lo      = np.clip(t_hat_eval_ds - q_hat, 0, None)
    hi      = t_hat_eval_ds + q_hat
    widths  = hi - lo
    obs     = ev_eval_ds == 1
    n_obs   = obs.sum()
    covered = ((dur_eval_ds[obs] >= lo[obs]) & (dur_eval_ds[obs] <= hi[obs])).sum()
    coverage = covered / n_obs if n_obs > 0 else np.nan
    print(f'alpha={alpha:.2f}  q_hat={q_hat:.4f} yr  '
          f'coverage={coverage:.1%} ({covered}/{n_obs})  '
          f'width={widths[0]:.3f} yr (constant)')
    results_cp_deepsurv[alpha] = dict(q_hat=q_hat, lo=lo, hi=hi, widths=widths,
                                       coverage=coverage, n_obs=n_obs, covered=covered,
                                       t_hat_eval=t_hat_eval_ds)

### 6.5 Agreement on predicted timing across models
Spearman correlation and mean absolute difference between the models' predicted median conversion times on the evaluation set (reported in Section 5.2).

In [ ]:
from scipy.stats import spearmanr
from itertools import combinations

pred_times = {
    'AFT':      results_cp_aft[0.20]['pred_median_eval'],
    'Cox PH':   results_cp_cox[0.20]['t_hat_eval'],
    'GBSA':     results_cp_gbsa[0.20]['t_hat_eval'],
    'DeepSurv': results_cp_deepsurv[0.20]['t_hat_eval'],
}

n_eval = len(pred_times['Cox PH'])
print(f'Comparing predicted median conversion time across all {n_eval} eval-set subjects '
      f'(observed-event and censored alike).\n')

# ── Pairwise Spearman correlation + mean absolute difference ────────────────
pair_rows = []
for m1, m2 in combinations(pred_times.keys(), 2):
    t1, t2 = pred_times[m1], pred_times[m2]
    rho, pval = spearmanr(t1, t2)
    mad = np.mean(np.abs(t1 - t2))
    max_diff = np.max(np.abs(t1 - t2))
    pair_rows.append(dict(pair=f'{m1} vs {m2}', spearman_rho=rho, spearman_p=pval,
                           mean_abs_diff_yr=mad, max_abs_diff_yr=max_diff))

pair_df = pd.DataFrame(pair_rows).sort_values('spearman_rho', ascending=False)
print('Pairwise agreement on predicted median conversion time:')
print(pair_df.to_string(index=False, float_format=lambda v: f'{v:.3f}'))

print(f'\nSummary across all 6 model pairs:')
print(f'  Spearman rho range : {pair_df["spearman_rho"].min():.3f} - {pair_df["spearman_rho"].max():.3f}')
print(f'  Mean abs diff range: {pair_df["mean_abs_diff_yr"].min():.2f} - {pair_df["mean_abs_diff_yr"].max():.2f} years')
print(f'  Largest single-subject disagreement: {pair_df["max_abs_diff_yr"].max():.2f} years '
      f'({pair_df.loc[pair_df["max_abs_diff_yr"].idxmax(), "pair"]})')

# ── Which pair agrees most/least tightly? ────────────────────────────────────
tightest = pair_df.iloc[0]
loosest  = pair_df.iloc[-1]
print(f'\nTightest-agreeing pair : {tightest["pair"]}  (rho={tightest["spearman_rho"]:.3f}, '
      f'mean abs diff={tightest["mean_abs_diff_yr"]:.2f} yr)')
print(f'Loosest-agreeing pair  : {loosest["pair"]}  (rho={loosest["spearman_rho"]:.3f}, '
      f'mean abs diff={loosest["mean_abs_diff_yr"]:.2f} yr)')

# ── Visualization: pairwise scatter grid of predicted times ─────────────────
model_names = list(pred_times.keys())
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for i, m_row in enumerate(model_names):
    for j, m_col in enumerate(model_names):
        ax = axes[i, j]
        if i == j:
            ax.hist(pred_times[m_row], bins=15, color='#2980b9', alpha=0.7)
            ax.set_title(m_row, fontsize=10)
        elif i > j:
            x, y = pred_times[m_col], pred_times[m_row]
            ax.scatter(x, y, s=12, alpha=0.5, color='#2980b9')
            lims = [0, max(x.max(), y.max()) * 1.05]
            ax.plot(lims, lims, color='#e74c3c', ls='--', lw=1, alpha=0.7)
            rho = spearmanr(x, y)[0]
            ax.text(0.05, 0.92, f'ρ={rho:.2f}', transform=ax.transAxes, fontsize=9,
                    va='top', color='#2c3e50')
        else:
            ax.axis('off')
        if i == 3:
            ax.set_xlabel(m_col, fontsize=9)
        if j == 0:
            ax.set_ylabel(m_row, fontsize=9)
plt.suptitle('Cross-Model Agreement on Predicted Median Conversion Time\n'
             '(diagonal: distribution per model; lower triangle: pairwise scatter, dashed = perfect agreement)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cross_model_time_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Landmark analysis (Section 5.4)
A Cox landmark supermodel refits the predictors at later visits (landmark times of 0 to 2 years) and predicts residual time to conversion, to test whether more longitudinal information reduces the bias. Fixed-width intervals at $\alpha = 0.20$.

### 7.1 Landmark dataset

In [ ]:
LANDMARK_TIMES = [0.0, 0.5, 1.0, 1.5, 2.0]  # years since baseline

# Survival labels under a separate name (surv_mci now holds DeepSurv's survival curves)
from preprocessing import build_survival_labels
surv_mci_labels = build_survival_labels(df, df_bl, 'MCI', 'AD',
                                         exclusion_rids=REVERSION_EXCLUSIONS)

# Reuse the baseline normalization constants so landmark features share the baseline scale
_mu_abeta, _sd_abeta = df_bl['ABETA'].mean(), df_bl['ABETA'].std()
_mu_av45,  _sd_av45  = df_bl['AV45'].mean(),  df_bl['AV45'].std()
_mu_tau,   _sd_tau   = df_bl['TAU'].mean(),   df_bl['TAU'].std()

def build_landmark_snapshot(df, surv, landmark_time):
    """
    For landmark time tau:
      1. Filter surv to subjects still at risk (duration > tau)
      2. Pull each at-risk subject's most recent visit with Years_bl <= tau
      3. Reconstruct the core_features feature set on that snapshot
      4. Return covariates + residual duration/event, tagged with landmark_time
    """
    at_risk_rids = surv.index[surv['duration'] > landmark_time]
    if len(at_risk_rids) == 0:
        return None

    snap = (
        df[(df['RID'].isin(at_risk_rids)) & (df['Years_bl'] <= landmark_time)]
        .sort_values(['RID', 'Years_bl'])
        .groupby('RID').tail(1)
        .copy()
    )
    snap = snap.set_index('RID').reindex(at_risk_rids).dropna(subset=['Years_bl'])

    if len(snap) == 0:
        return None

    # ── Engineered features (as in Section 3.4) ──────
    # ICV ratios are rescaled by 100 so the landmark Cox coefficients are numerically stable.
    ICV_SCALE = 100
    for region in ['Hippocampus', 'Entorhinal', 'Ventricles', 'Fusiform', 'MidTemp', 'WholeBrain']:
        snap[f'{region}_ICV'] = (snap[region] / snap['ICV']) * ICV_SCALE
    snap['hippo_entorhinal_ratio'] = snap['Hippocampus_ICV'] / snap['Entorhinal_ICV']
    snap['midtemp_fusiform_ratio'] = snap['MidTemp_ICV'] / snap['Fusiform_ICV']

    snap['ABETA_z'] = (snap['ABETA'] - _mu_abeta) / (_sd_abeta + 1e-9)
    snap['AV45_z']  = (snap['AV45']  - _mu_av45)  / (_sd_av45  + 1e-9)
    snap['ABETA_z_flipped'] = -snap['ABETA_z']
    snap['amyloid_composite'] = snap[['ABETA_z_flipped', 'AV45_z']].mean(axis=1, skipna=True)
    snap['amyloid_positive'] = (snap['ABETA'] < 880).astype(float)
    _fallback = snap['ABETA'].isna() & snap['AV45'].notna()
    snap.loc[_fallback, 'amyloid_positive'] = (snap.loc[_fallback, 'AV45'] > 1.11).astype(float)

    snap['TAU_z'] = (snap['TAU'] - _mu_tau) / (_sd_tau + 1e-9)
    snap['PTGENDER_num'] = (snap['PTGENDER'] == 'Male').astype(int)

    snap['APOE4_x_amyloid']   = snap['APOE4'] * snap['amyloid_composite'].fillna(0)
    snap['APOE4_x_tau']       = snap['APOE4'] * snap['TAU_z'].fillna(0)
    snap['APOE4_x_hippo_ICV'] = snap['APOE4'] * snap['Hippocampus_ICV'].fillna(
        snap['Hippocampus_ICV'].median())
    snap['APOE4_amyloid_pos'] = snap['APOE4'] * snap['amyloid_positive'].fillna(0)

    # ── Residual outcome from this landmark ───────────────────────────────────
    snap['duration_resid'] = surv.loc[snap.index, 'duration'] - landmark_time
    snap['event']          = surv.loc[snap.index, 'event']
    snap['landmark_time']  = landmark_time
    snap['n_visits_used']  = df[df['RID'].isin(snap.index) & (df['Years_bl'] <= landmark_time)] \
                                .groupby('RID').size().reindex(snap.index).values
    snap['RID']            = snap.index

    return snap[core_features + ['duration_resid', 'event', 'landmark_time', 'n_visits_used', 'RID']]

landmark_frames = [build_landmark_snapshot(df, surv_mci_labels, tau) for tau in LANDMARK_TIMES]
landmark_stack = pd.concat([f for f in landmark_frames if f is not None], ignore_index=True)

print(f'Landmark stack: {len(landmark_stack)} (subject, landmark) rows '
      f'from {landmark_stack["RID"].nunique()} unique subjects across {len(LANDMARK_TIMES)} landmarks')
print(landmark_stack.groupby('landmark_time').agg(
    n_subjects=('RID', 'nunique'), n_events=('event', 'sum'),
    mean_visits=('n_visits_used', 'mean')))

### 7.2 Landmark supermodel
Subjects (not rows) are split into training, calibration, and evaluation sets, so a subject's landmarks never straddle two sets.

In [ ]:
from lifelines import CoxPHFitter
from sklearn.model_selection import train_test_split as _lm_tts

# Split at the SUBJECT level (not row level), the same subject appears at
# multiple landmarks, so splitting rows would leak a person's data across
# cal/eval, breaking the exchangeability conformal prediction depends on.
unique_rids = landmark_stack['RID'].unique()
rid_events = surv_mci_labels.loc[unique_rids, 'event']
lm_train_rids, lm_caleval_rids = _lm_tts(
    unique_rids, test_size=0.30, random_state=RANDOM_SEED, stratify=rid_events)
lm_cal_rids, lm_eval_rids = _lm_tts(
    lm_caleval_rids, test_size=0.333, random_state=RANDOM_SEED,
    stratify=surv_mci_labels.loc[lm_caleval_rids, 'event'])

lm_train = landmark_stack[landmark_stack['RID'].isin(lm_train_rids)].copy()
lm_cal   = landmark_stack[landmark_stack['RID'].isin(lm_cal_rids)].copy()
lm_eval  = landmark_stack[landmark_stack['RID'].isin(lm_eval_rids)].copy()

print(f'Landmark train: {len(lm_train)} rows / {lm_train["RID"].nunique()} subjects')
print(f'Landmark cal  : {len(lm_cal)} rows / {lm_cal["RID"].nunique()} subjects')
print(f'Landmark eval : {len(lm_eval)} rows / {lm_eval["RID"].nunique()} subjects')

# Median imputation with training medians
for col in core_features:
    med = lm_train[col].median()
    lm_train[col] = lm_train[col].fillna(med)
    lm_cal[col]   = lm_cal[col].fillna(med)
    lm_eval[col]  = lm_eval[col].fillna(med)

# ── Fit landmark supermodel Cox PH ────────────────────────────────────────────
# landmark_time included as a covariate (van Houwelingen's landmark supermodel).
# cluster_col='RID' accounts for within subject correlation across landmarks;
# falls back to no clustering if the installed lifelines version doesn't
# support it, point estimates/predictions are unaffected either way, only
# standard error estimates would be slightly anti conservative without it.
cox_fit_df = lm_train[core_features + ['duration_resid', 'event', 'landmark_time', 'RID']].copy()
cox_lm = CoxPHFitter(penalizer=0.1)
try:
    cox_lm.fit(cox_fit_df, duration_col='duration_resid', event_col='event',
               cluster_col='RID')
    print('Fit with cluster_col=RID (robust SEs for within-subject correlation).')
except TypeError:
    cox_lm.fit(cox_fit_df.drop(columns='RID'), duration_col='duration_resid',
               event_col='event')
    print('cluster_col not supported by this lifelines version: fit without it '
          '(point estimates/predictions valid; SEs may be mildly anti-conservative).')

print(cox_lm.print_summary())

### 7.3 Mean $\pi$ by landmark time (single split)

In [ ]:
lm_surv_cal_matrices = {}
lm_results_by_landmark = {}

for tau in LANDMARK_TIMES:
    cal_tau  = lm_cal[lm_cal['landmark_time'] == tau]
    eval_tau = lm_eval[lm_eval['landmark_time'] == tau]
    if len(cal_tau) < 5 or len(eval_tau) < 5:
        print(f'tau={tau}: insufficient cal/eval rows ({len(cal_tau)}/{len(eval_tau)}), skipping.')
        continue

    # Predicted median residual time via survival function inversion at p=0.5
    surv_cal_tau  = cox_lm.predict_survival_function(cal_tau[core_features + ['landmark_time']])
    surv_eval_tau = cox_lm.predict_survival_function(eval_tau[core_features + ['landmark_time']])
    t_hat_cal_tau  = _invert_surv(surv_cal_tau)
    t_hat_eval_tau = _invert_surv(surv_eval_tau)

    dur_cal_tau  = cal_tau['duration_resid'].values
    dur_eval_tau = eval_tau['duration_resid'].values
    ev_cal_tau   = cal_tau['event'].values
    ev_eval_tau  = eval_tau['event'].values

    obs_mask = ev_cal_tau == 1
    if obs_mask.sum() < 3:
        print(f'tau={tau}: fewer than 3 observed events in cal, skipping.')
        continue
    scores_tau = t_hat_cal_tau[obs_mask] - dur_cal_tau[obs_mask]
    q_hat_tau  = _conformal_quantile(scores_tau, 0.20, obs_mask.sum())  # alpha = 0.20

    lo = np.clip(t_hat_eval_tau - q_hat_tau, 0, None)
    hi = t_hat_eval_tau + q_hat_tau
    obs_eval = ev_eval_tau == 1
    n_obs = obs_eval.sum()
    if n_obs == 0:
        print(f'tau={tau}: no observed events in eval, skipping bias check.')
        continue
    covered = ((dur_eval_tau[obs_eval] >= lo[obs_eval]) & (dur_eval_tau[obs_eval] <= hi[obs_eval])).sum()
    coverage = covered / n_obs

    width = hi[obs_eval] - lo[obs_eval]
    valid = width > 0
    pct = (dur_eval_tau[obs_eval][valid] - lo[obs_eval][valid]) / width[valid]
    pct = np.clip(pct, 0, 1)

    lm_results_by_landmark[tau] = dict(coverage=coverage, n_obs=n_obs,
                                        mean_pct=pct.mean(), median_pct=np.median(pct))
    print(f'tau={tau:.1f}yr | n_eval_events={n_obs} | coverage={coverage:.1%} | mean_pi={pct.mean():.3f}')

### 7.4 Sensitivity to visit count

In [ ]:
MIN_VISIT_THRESHOLDS = [1, 2, 3, 4]

def run_landmark_pipeline_for_min_visits(min_visits):
    """Restricts landmark_stack to (subject, landmark) rows with at least
    min_visits visits used in the snapshot, then refits + re-evaluates."""
    stack = landmark_stack[landmark_stack['n_visits_used'] >= min_visits].copy()
    if stack['RID'].nunique() < 20:
        return None  # too few subjects left to be meaningful

    rids = stack['RID'].unique()
    events = surv_mci_labels.loc[rids, 'event']
    train_rids, caleval_rids = train_test_split(
        rids, test_size=0.30, random_state=RANDOM_SEED, stratify=events)
    cal_rids, eval_rids = train_test_split(
        caleval_rids, test_size=0.333, random_state=RANDOM_SEED,
        stratify=surv_mci_labels.loc[caleval_rids, 'event'])

    train = stack[stack['RID'].isin(train_rids)].copy()
    cal   = stack[stack['RID'].isin(cal_rids)].copy()
    ev_df = stack[stack['RID'].isin(eval_rids)].copy()
    for col in core_features:
        med = train[col].median()
        train[col] = train[col].fillna(med)
        cal[col]   = cal[col].fillna(med)
        ev_df[col] = ev_df[col].fillna(med)

    model = CoxPHFitter(penalizer=0.1)
    try:
        model.fit(train[core_features + ['duration_resid', 'event', 'landmark_time', 'RID']],
                  duration_col='duration_resid', event_col='event', cluster_col='RID')
    except TypeError:
        model.fit(train[core_features + ['duration_resid', 'event', 'landmark_time']],
                  duration_col='duration_resid', event_col='event')

    obs_cal = cal['event'].values == 1
    if obs_cal.sum() < 3:
        return None
    t_hat_cal = _invert_surv(model.predict_survival_function(cal[core_features + ['landmark_time']]))
    t_hat_eval = _invert_surv(model.predict_survival_function(ev_df[core_features + ['landmark_time']]))
    scores = t_hat_cal[obs_cal] - cal['duration_resid'].values[obs_cal]
    q_hat = _conformal_quantile(scores, 0.20, obs_cal.sum())

    lo = np.clip(t_hat_eval - q_hat, 0, None)
    hi = t_hat_eval + q_hat
    obs_eval = ev_df['event'].values == 1
    if obs_eval.sum() == 0:
        return None
    width = hi[obs_eval] - lo[obs_eval]
    valid = width > 0
    true_t = ev_df['duration_resid'].values[obs_eval][valid]
    pct = np.clip((true_t - lo[obs_eval][valid]) / width[valid], 0, 1)

    return dict(min_visits=min_visits, n_subjects=stack['RID'].nunique(),
                n_rows=len(stack), n_eval_events=obs_eval.sum(),
                mean_pi=pct.mean(), median_pi=np.median(pct))

visit_sensitivity_results = [
    r for r in (run_landmark_pipeline_for_min_visits(m) for m in MIN_VISIT_THRESHOLDS) if r
]
visit_sensitivity_df = pd.DataFrame(visit_sensitivity_results)
print(visit_sensitivity_df.to_string(index=False))

### 7.5 Five-fold pooling with bootstrap CIs
The landmark analysis repeated over five folds, pooling the evaluation conversions at each landmark time, with 1,000-resample bootstrap 95% confidence intervals.

In [ ]:
RUN_KFOLD_BOOTSTRAP = True   # refits the landmark Cox model 5 times (fast)

N_FOLDS = 5
BOOTSTRAP_ITERS = 1000

def bootstrap_ci(values, n_iter=BOOTSTRAP_ITERS, seed=RANDOM_SEED):
    values = np.array(values)
    rng = np.random.default_rng(seed)
    boot_means = np.array([
        rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_iter)
    ])
    return values.mean(), np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)

if RUN_KFOLD_BOOTSTRAP:
    from sklearn.model_selection import StratifiedKFold, train_test_split as _kf_tts

    unique_rids_arr = landmark_stack['RID'].unique()
    fold_events = surv_mci_labels.loc[unique_rids_arr, 'event'].values

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    pooled_pct_by_tau = {tau: [] for tau in LANDMARK_TIMES}
    fold_coverage_records = []

    for fold_idx, (trainval_idx, eval_idx) in enumerate(skf.split(unique_rids_arr, fold_events)):
        trainval_rids = unique_rids_arr[trainval_idx]
        eval_rids     = unique_rids_arr[eval_idx]

        trainval_events = surv_mci_labels.loc[trainval_rids, 'event']
        train_rids, cal_rids = _kf_tts(
            trainval_rids, test_size=0.20, random_state=RANDOM_SEED + fold_idx,
            stratify=trainval_events)

        train_fold = landmark_stack[landmark_stack['RID'].isin(train_rids)].copy()
        cal_fold   = landmark_stack[landmark_stack['RID'].isin(cal_rids)].copy()
        eval_fold  = landmark_stack[landmark_stack['RID'].isin(eval_rids)].copy()

        for col in core_features:
            med = train_fold[col].median()
            train_fold[col] = train_fold[col].fillna(med)
            cal_fold[col]   = cal_fold[col].fillna(med)
            eval_fold[col]  = eval_fold[col].fillna(med)

        model = CoxPHFitter(penalizer=0.1)
        try:
            model.fit(train_fold[core_features + ['duration_resid', 'event', 'landmark_time', 'RID']],
                       duration_col='duration_resid', event_col='event', cluster_col='RID')
        except TypeError:
            model.fit(train_fold[core_features + ['duration_resid', 'event', 'landmark_time']],
                       duration_col='duration_resid', event_col='event')

        for tau in LANDMARK_TIMES:
            cal_tau  = cal_fold[cal_fold['landmark_time'] == tau]
            eval_tau = eval_fold[eval_fold['landmark_time'] == tau]
            if len(cal_tau) < 5 or len(eval_tau) < 5:
                continue

            obs_cal = cal_tau['event'].values == 1
            if obs_cal.sum() < 3:
                continue

            t_hat_cal  = _invert_surv(model.predict_survival_function(cal_tau[core_features + ['landmark_time']]))
            t_hat_eval = _invert_surv(model.predict_survival_function(eval_tau[core_features + ['landmark_time']]))
            dur_cal  = cal_tau['duration_resid'].values
            dur_eval = eval_tau['duration_resid'].values

            scores = t_hat_cal[obs_cal] - dur_cal[obs_cal]
            q_hat = _conformal_quantile(scores, 0.20, obs_cal.sum())

            lo = np.clip(t_hat_eval - q_hat, 0, None)
            hi = t_hat_eval + q_hat
            obs_eval = eval_tau['event'].values == 1
            if obs_eval.sum() == 0:
                continue

            covered = ((dur_eval[obs_eval] >= lo[obs_eval]) & (dur_eval[obs_eval] <= hi[obs_eval])).sum()
            width = hi[obs_eval] - lo[obs_eval]
            valid = width > 0
            pct = np.clip((dur_eval[obs_eval][valid] - lo[obs_eval][valid]) / width[valid], 0, 1)

            pooled_pct_by_tau[tau].extend(pct.tolist())
            fold_coverage_records.append(dict(fold=fold_idx, tau=tau,
                                               n_eval_events=obs_eval.sum(),
                                               coverage=covered / obs_eval.sum()))
        print(f'Fold {fold_idx + 1}/{N_FOLDS} done.')

    fold_df = pd.DataFrame(fold_coverage_records)
    print('\nPooled eval events per landmark (sum across all 5 folds):')
    print(fold_df.groupby('tau')['n_eval_events'].sum())

    # ── Bootstrap CI per landmark ──────────────────────────────────────────────
    bootstrap_results = {}
    for tau in LANDMARK_TIMES:
        vals = pooled_pct_by_tau[tau]
        if len(vals) < 5:
            print(f'tau={tau}: insufficient pooled events ({len(vals)}), skipping.')
            continue
        mean_pi, ci_lo, ci_hi = bootstrap_ci(vals)
        bootstrap_results[tau] = dict(n=len(vals), mean_pi=mean_pi, ci_lo=ci_lo, ci_hi=ci_hi)
        print(f'tau={tau:.1f}yr | n_pooled_events={len(vals)} | mean_pi={mean_pi:.3f} '
              f'| 95% CI=[{ci_lo:.3f}, {ci_hi:.3f}]')
else:
    print('Skipped (RUN_KFOLD_BOOTSTRAP = False).')

### 7.6 Table 3: landmark results
Five-fold pooled mean $\pi$ by landmark time with bootstrap 95% CIs.

In [ ]:
lm = pd.DataFrame([dict(tau=t, n_events=int(v['n']), mean_pi=float(v['mean_pi']),
                        ci_lo=float(v['ci_lo']), ci_hi=float(v['ci_hi']))
                   for t, v in sorted(bootstrap_results.items())])
lm.to_csv(OUT_DIR / 'landmark_pi_by_tau.csv', index=False)
print(lm.round(3).to_string(index=False))
print('\nTable 3 rows:')
for r in lm.itertuples():
    print(f'{r.tau:.1f} & {r.n_events} & {r.mean_pi:.3f} & [{r.ci_lo:.3f}, {r.ci_hi:.3f}] \\\\')

## 8. Censoring (Section 5.5)
About 60% of the cohort is censored, and $\pi$ is computed on observed conversions only, so later converters may be under-represented. The censoring survival function $\hat G(t)$ is fit by Kaplan-Meier on the training subjects, and each calibration conversion is weighted by $1/\hat G(T_i)$ (clipped at $\hat G \ge 0.05$).

### 8.1 Censoring distribution
Also tests whether censoring depends on the covariates: a Cox model with censoring as the event, with Benjamini-Hochberg correction.

In [ ]:
from lifelines import KaplanMeierFitter, CoxPHFitter as _CoxPHFitter_diag

# ── Marginal censoring survival curve G(t) = P(remain uncensored past t) ────
# Flip the event indicator: "event" here means CENSORED, not converted.
kmf_censoring = KaplanMeierFitter()
_cens_train = surv_mci_labels.loc[surv_mci_labels.index.isin(MCI_SPLIT['train'])]  # training subjects only
print(f'Censoring curve fit on {len(_cens_train)} training subjects')
kmf_censoring.fit(_cens_train['duration'],
                   event_observed=(1 - _cens_train['event']),
                   label='Censoring survival function G(t)')

fig, ax = plt.subplots(figsize=(8, 5))
kmf_censoring.plot_survival_function(ax=ax, color='#8e44ad', ci_show=True, ci_alpha=0.15)
ax.set(xlabel='Years from baseline', ylabel='P(remain uncensored past t)',
       title='Marginal Censoring Survival Function: G(t)\n(MCI → AD cohort)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'ipcw_censoring_curve.png', dpi=150, bbox_inches='tight')
plt.show()

def G_hat(t, kmf=kmf_censoring):
    """Evaluate the censoring survival function at time(s) t."""
    t = np.atleast_1d(t)
    return np.array([
        kmf.survival_function_at_times(tt).values[0] if tt <= kmf.timeline.max()
        else kmf.survival_function_.iloc[-1, 0]
        for tt in t
    ])

# ── Informativeness diagnostic: is censoring predictable from covariates? ───
# Cox model with censoring as the event. If no feature is significant after correction,
# a marginal G(t) (not depending on covariates) is justified.
baseline_snap = build_landmark_snapshot(df, surv_mci_labels, 0.0)
diag_df = baseline_snap[core_features].copy()
for col in core_features:
    diag_df[col] = diag_df[col].fillna(diag_df[col].median())
diag_df['duration'] = surv_mci_labels.loc[baseline_snap.index, 'duration']
diag_df['censored'] = 1 - surv_mci_labels.loc[baseline_snap.index, 'event']

cox_diag = _CoxPHFitter_diag(penalizer=0.1)
cox_diag.fit(diag_df, duration_col='duration', event_col='censored')
diag_summary = cox_diag.summary[['coef', 'exp(coef)', 'p']].sort_values('p')
n_significant_uncorrected = (diag_summary['p'] < 0.05).sum()

# ── Multiple-testing correction (Benjamini-Hochberg) ──────────────────────────
from statsmodels.stats.multitest import multipletests
bh_reject, bh_pvals_corrected, _, _ = multipletests(diag_summary['p'].values, alpha=0.05, method='fdr_bh')
diag_summary['p_bh_corrected'] = bh_pvals_corrected
diag_summary['significant_bh'] = bh_reject
n_significant_bh = bh_reject.sum()

print(f'Censoring-informativeness diagnostic: {n_significant_uncorrected} / {len(diag_summary)} '
      f'features significant at uncorrected p<0.05.')
print(f'After Benjamini-Hochberg FDR correction: {n_significant_bh} / {len(diag_summary)} remain significant.')
print(diag_summary.head(10).to_string())
if n_significant_bh == 0:
    print('\nNo feature is significant after correction: a marginal G(t) is justified.')
else:
    print(f'\n{n_significant_bh} feature(s) significant after correction: censoring may depend on covariates.')

### 8.2 Reweighted calibration, Cox PH
The calibration quantile $\hat q$ is recomputed with IPCW weights. This changes the interval width but not its center.

In [ ]:
def weighted_quantile(values, weights, quantile):
    """Weighted empirical quantile (interpolated)."""
    values = np.asarray(values)
    weights = np.asarray(weights)
    sorter = np.argsort(values)
    values, weights = values[sorter], weights[sorter]
    cum_weights = np.cumsum(weights) - 0.5 * weights
    cum_weights /= np.sum(weights)
    return np.interp(quantile, cum_weights, values)

_, surv_cox_cal_fresh  = calc_cox_ph_c(cox_model_mci, cox_scaler_mci, X_mci_cal_cox,
                                        y_ev_mci_cal,  y_dur_mci_cal)
_, surv_cox_eval_fresh = calc_cox_ph_c(cox_model_mci, cox_scaler_mci, X_mci_eval_cox,
                                        y_ev_mci_eval, y_dur_mci_eval)

t_hat_cal_fresh  = _invert_surv(surv_cox_cal_fresh)
t_hat_eval_fresh = _invert_surv(surv_cox_eval_fresh)

dur_cal_fresh  = np.array(y_dur_mci_cal,  dtype=np.float64)
ev_cal_fresh   = np.array(y_ev_mci_cal,   dtype=np.int32)
dur_eval_fresh = np.array(y_dur_mci_eval, dtype=np.float64)
ev_eval_fresh  = np.array(y_ev_mci_eval,  dtype=np.int32)

obs_cal_fresh    = ev_cal_fresh == 1
scores_cal_fresh = t_hat_cal_fresh[obs_cal_fresh] - dur_cal_fresh[obs_cal_fresh]
n_cal_fresh      = obs_cal_fresh.sum()

print(f'Rebuilt Cox cal/eval: cal n={len(dur_cal_fresh)} (events={obs_cal_fresh.sum()}), '
      f'eval n={len(dur_eval_fresh)} (events={ev_eval_fresh.sum()})')

G_at_events = G_hat(dur_cal_fresh[obs_cal_fresh])
G_at_events = np.clip(G_at_events, 0.05, 1.0)  # avoid exploding weights near end of follow-up
ipcw_weights = 1.0 / G_at_events

print(f'IPCW weight range: min={ipcw_weights.min():.2f}, max={ipcw_weights.max():.2f}, '
      f'mean={ipcw_weights.mean():.2f}')

results_ipcw_cox = {}
for alpha in [0.10, 0.20]:
    q_hat_unweighted = _conformal_quantile(scores_cal_fresh, alpha, n_cal_fresh)
    q_hat_ipcw = weighted_quantile(scores_cal_fresh, ipcw_weights, 1 - alpha)

    for label, q_hat in [('unweighted', q_hat_unweighted), ('IPCW', q_hat_ipcw)]:
        lo = np.clip(t_hat_eval_fresh - q_hat, 0, None)
        hi = t_hat_eval_fresh + q_hat
        obs = ev_eval_fresh == 1
        n_obs = obs.sum()
        covered = ((dur_eval_fresh[obs] >= lo[obs]) & (dur_eval_fresh[obs] <= hi[obs])).sum()
        coverage = covered / n_obs

        width = hi[obs] - lo[obs]
        valid = width > 0
        pct = np.clip((dur_eval_fresh[obs][valid] - lo[obs][valid]) / width[valid], 0, 1)

        results_ipcw_cox[(alpha, label)] = dict(q_hat=q_hat, coverage=coverage,
                                                  mean_pi=pct.mean(), n_obs=n_obs)
        print(f'alpha={alpha:.2f} [{label:>11}] q_hat={q_hat:.4f}yr coverage={coverage:.1%} '
              f'mean_pi={pct.mean():.3f}')
    print()

### 8.3 Reweighted calibration, all four models
Mean $\pi$ at $\alpha = 0.20$ under unweighted and IPCW-reweighted fixed-width calibration.

In [ ]:
dur_cal_all  = np.array(y_dur_mci_cal,  dtype=np.float64)
ev_cal_all   = np.array(y_ev_mci_cal,   dtype=np.int32)
dur_eval_all = np.array(y_dur_mci_eval, dtype=np.float64)
ev_eval_all  = np.array(y_ev_mci_eval,  dtype=np.int32)

def fixed_width_mean_pi(surv_cal, surv_eval, alpha=0.20, weighted=False):
    """Mean within-interval percentile of fixed-width intervals, with the calibration
    quantile either unweighted or IPCW-weighted."""
    t_hat_cal, t_hat_eval = _invert_surv(surv_cal), _invert_surv(surv_eval)
    obs_cal = ev_cal_all == 1
    scores = t_hat_cal[obs_cal] - dur_cal_all[obs_cal]
    if weighted:
        w = 1.0 / np.clip(G_hat(dur_cal_all[obs_cal]), 0.05, 1.0)
        q_hat = weighted_quantile(scores, w, 1 - alpha)
    else:
        q_hat = _conformal_quantile(scores, alpha, obs_cal.sum())
    lo, hi = np.clip(t_hat_eval - q_hat, 0, None), t_hat_eval + q_hat
    obs = ev_eval_all == 1
    width = hi[obs] - lo[obs]
    ok = width > 0
    return np.clip((dur_eval_all[obs][ok] - lo[obs][ok]) / width[ok], 0, 1).mean()

_, surv_gbsa_cal_fresh  = calc_gbsa_c(gbsa_model_mci, X_mci_cal,  y_ev_mci_cal,  y_dur_mci_cal)
_, surv_gbsa_eval_fresh = calc_gbsa_c(gbsa_model_mci, X_mci_eval, y_ev_mci_eval, y_dur_mci_eval)
aft_surv_cal_fresh  = aft_model_mci.predict_survival_function(X_mci_cal)
aft_surv_eval_fresh = aft_model_mci.predict_survival_function(X_mci_eval)
_, surv_ds_cal_fresh  = calc_deepsurv_c(ds_model_mci, ds_scaler_mci, X_mci_cal,  y_ev_mci_cal,  y_dur_mci_cal)
_, surv_ds_eval_fresh = calc_deepsurv_c(ds_model_mci, ds_scaler_mci, X_mci_eval, y_ev_mci_eval, y_dur_mci_eval)

ipcw_calibration_results = {
    'Cox PH': dict(unweighted=results_ipcw_cox[(0.20, 'unweighted')]['mean_pi'],
                   ipcw=results_ipcw_cox[(0.20, 'IPCW')]['mean_pi']),
}
for name, (s_cal, s_eval) in {'GBSA': (surv_gbsa_cal_fresh, surv_gbsa_eval_fresh),
                              'AFT': (aft_surv_cal_fresh, aft_surv_eval_fresh),
                              'DeepSurv': (surv_ds_cal_fresh, surv_ds_eval_fresh)}.items():
    ipcw_calibration_results[name] = dict(unweighted=fixed_width_mean_pi(s_cal, s_eval),
                                          ipcw=fixed_width_mean_pi(s_cal, s_eval, weighted=True))

ipcw_calibration_df = pd.DataFrame(ipcw_calibration_results).T
ipcw_calibration_df['change'] = ipcw_calibration_df['ipcw'] - ipcw_calibration_df['unweighted']
print('Mean pi at alpha = 0.20, unweighted vs IPCW-reweighted calibration:')
print(ipcw_calibration_df.round(3).to_string())

## 9. Adaptive intervals (Sections 3.3, 5.5, and 5.6)
CSD and CiPOT calibrate each patient's whole predicted survival curve on the calibration conversions. For each level $\rho \in \{0.05, 0.10, 0.20, \ldots, 0.90, 0.95\}$, $\hat t_i(\rho)$ is the first time $\hat S_i(t) \le \rho$. CSD shifts each percentile time by the calibrated residual quantile and rearranges the result to be monotone. CiPOT reads each patient's curve at a recalibrated probability level, using scores $r_i = \hat S_i(T_i)$. The interval at $\alpha$ runs from the $\rho = 1-\alpha/2$ time to the $\rho = \alpha/2$ time.

### 9.1 Shared functions

In [ ]:
# Percentile levels rho (survival probabilities); rho = 0.1 is the 90th percentile of event time
percentile_levels = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

def compute_pcts(surv_matrix, rho_levels):
    """
    Compute Percentile Times (PCTs) for each subject and each rho level.
    PCT(rho, i) = first time t where S(t | x_i) <= rho
    Returns array of shape (n_subjects, n_rho_levels)
    """
    times = surv_matrix.index.values.astype(np.float64)
    n_subjects = surv_matrix.shape[1]
    pcts = np.full((n_subjects, len(rho_levels)), times[-1])
    for j in range(n_subjects):
        curve = surv_matrix.iloc[:, j].values
        for k, rho in enumerate(rho_levels):
            below = np.where(curve <= rho)[0]
            if len(below) > 0:
                pcts[j, k] = times[below[0]]
    return pcts

def csd_calibrate(pcts_cal, dur_cal, ev_cal, rho_levels):
    """
    Compute per percentile conformal corrections using observed events only.
    Returns q_hat array of shape (n_rho_levels,)
    """
    obs_mask = ev_cal == 1
    pcts_obs  = pcts_cal[obs_mask]       # shape: (n_obs, n_rho)
    dur_obs   = dur_cal[obs_mask]        # shape: (n_obs,)
    n_obs     = obs_mask.sum()

    q_hats = np.zeros(len(rho_levels))
    for k, rho in enumerate(rho_levels):
        scores = pcts_obs[:, k] - dur_obs   # predicted - actual
        level  = min(np.ceil((n_obs + 1) * rho) / n_obs, 1.0)
        q_hats[k] = np.quantile(scores, level)

    return q_hats

def csd_adjust_pcts(pcts_eval, q_hats):
    """
    Apply per percentile corrections to evaluation set PCTs.
    Returns adjusted PCTs of shape (n_eval_subjects, n_rho_levels)
    """
    return pcts_eval - q_hats[np.newaxis, :]   # broadcast correction

def enforce_monotonic_pcts(pcts_adj):
    """Monotone rearrangement (Chernozhukov et al.): columns run from rho = 0.1 (late)
    to rho = 0.9 (early), so each row must be non-increasing."""
    return np.sort(pcts_adj, axis=1)[:, ::-1]

# AFT survival curves on the evaluation set
_test_eval_aft = X_mci_eval.copy().reset_index(drop=True)
aft_surv_eval  = aft_model_mci.predict_survival_function(_test_eval_aft)

### 9.2 Table 5 and Figure 4: fixed width, CSD, and CiPOT
Mean $\pi$, coverage, misses below and above the interval, and median widths for every model at both $\alpha$ levels. Figure 4 shows the distribution of $\pi$ for Cox PH; Supplementary Figure S1 shows all four models. Results are saved to `outputs/three_methods_summary.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RHO_EXT = np.array([0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95])
ALPHAS = {0.10: (0.95, 0.05), 0.20: (0.90, 0.10)}   # alpha -> (rho for lower end, rho for upper end)
COL = {r: k for k, r in enumerate(RHO_EXT)}
METHODS = ['Fixed width', 'CSD', 'CiPOT']

MODELS = {   # name: (surv_cal, surv_eval, fixed-width CP results, eval event, eval duration)
    'Cox PH':        (surv_cox_mci,  surv_cox_mci_eval,  results_cp_cox,      ev_eval_cox,  dur_eval_cox),
    'AFT (Weibull)': (aft_surv_mci,  aft_surv_eval,      results_cp_aft,      ev_eval_aft,  dur_eval_aft),
    'GBSA':          (surv_gbsa_mci, surv_gbsa_mci_eval, results_cp_gbsa,     ev_eval_gbsa, dur_eval_gbsa),
    'DeepSurv':      (surv_mci,      surv_mci_eval,      results_cp_deepsurv, ev_eval_ds,   dur_eval_ds),
}
dur_cal_all = np.asarray(y_dur_mci_cal, dtype=float)
ev_cal_all = np.asarray(y_ev_mci_cal, dtype=int)


def _finite_level(rho, n):
    return min(np.ceil((n + 1) * rho) / n, 1.0)


def _surv_at(surv_matrix, times_obs):
    """Predicted survival probability of each subject at its own observed time (step function)."""
    grid = surv_matrix.index.values.astype(float)
    vals = surv_matrix.values
    idx = np.searchsorted(grid, np.asarray(times_obs, dtype=float), side='right') - 1
    out = np.ones(len(times_obs))
    has = idx >= 0
    out[has] = vals[idx[has], np.arange(len(times_obs))[has]]
    return out


def _pi_and_summary(lo, hi, dur, ev):
    """Per-subject pi for observed events, plus coverage, miss direction, and widths."""
    lo, hi, dur, ev = (np.asarray(a, dtype=float) for a in (lo, hi, dur, ev))
    w_all = hi - lo                                   # widths for every eval subject
    o = ev == 1
    lo_o, hi_o, t = lo[o], hi[o], dur[o]
    cov = (t >= lo_o) & (t <= hi_o)
    w = hi_o - lo_o
    ok = w > 0
    pi = np.clip((t[ok] - lo_o[ok]) / w[ok], 0, 1)
    summ = dict(n_obs=int(o.sum()),
                coverage_pct=round(100 * float(cov.mean()), 1),
                mean_pi=round(float(pi.mean()), 3),
                miss_below=int((t < lo_o).sum()),
                miss_above=int((t > hi_o).sum()),
                median_width_yr=round(float(np.median(w_all)), 2),
                mean_width_yr=round(float(np.mean(w_all)), 2))
    return pi, summ


rows, PI = [], {}   # PI[(model, alpha, method)] = per-subject pi array
for name, (s_cal, s_eval, R_fixed, ev_e, dur_e) in MODELS.items():
    # CSD on the extended grid
    q_csd = csd_calibrate(compute_pcts(s_cal, RHO_EXT), dur_cal_all, ev_cal_all, RHO_EXT)
    pcts_csd = enforce_monotonic_pcts(np.clip(csd_adjust_pcts(compute_pcts(s_eval, RHO_EXT), q_csd), 0, None))
    # Simplified CiPOT: recalibrated probability level = rho-quantile of S_i(T_i) over calibration events
    o_cal = ev_cal_all == 1
    ipot = _surv_at(s_cal, dur_cal_all)[o_cal]
    p_levels = np.array([np.quantile(ipot, _finite_level(r, len(ipot))) for r in RHO_EXT])
    pcts_cipot = enforce_monotonic_pcts(compute_pcts(s_eval, p_levels))

    for a, (r_lo, r_hi) in ALPHAS.items():
        bounds = {'Fixed width': (R_fixed[a]['lo'], R_fixed[a]['hi']),
                  'CSD': (pcts_csd[:, COL[r_lo]], pcts_csd[:, COL[r_hi]]),
                  'CiPOT': (pcts_cipot[:, COL[r_lo]], pcts_cipot[:, COL[r_hi]])}
        for m in METHODS:
            pi, summ = _pi_and_summary(*bounds[m], dur_e, ev_e)
            PI[(name, a, m)] = pi
            rows.append(dict(model=name, alpha=a, method=m, **summ))

review_df = pd.DataFrame(rows)
review_df.to_csv(OUT_DIR / 'three_methods_summary.csv', index=False)
pd.set_option('display.width', 200)
print(review_df.to_string(index=False))
print('\nsaved', OUT_DIR / 'three_methods_summary.csv')

# ── Summary ──────────────────────────────────────────────────────────────────
print('\nMean pi (coverage %):')
for name in MODELS:
    for a in ALPHAS:
        cells = []
        for m in METHODS:
            r = review_df[(review_df.model == name) & (review_df.alpha == a) & (review_df.method == m)].iloc[0]
            cells.append(f'{r.mean_pi:.3f} ({r.coverage_pct:.1f})')
        print(f'{name:<14} {a:.2f} & ' + ' & '.join(cells) + r' \\')

print('\nMisses pooled across the four models:')
pooled = review_df.groupby(['alpha', 'method'])[['miss_below', 'miss_above']].sum()
print(pooled.to_string())

print('\n=== Median interval width, years (optional sentence in 5.6) ===')
print(review_df.pivot_table(index=['model', 'alpha'], columns='method',
                            values='median_width_yr').round(2).to_string())

# ── Figures ──────────────────────────────────────────────────────────────────
PAPER_FIG_DIR = FIG_DIR / 'paper'
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
RC = {'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7.5, 'legend.fontsize': 6.5,
      'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'axes.linewidth': 0.7}
METHOD_COLORS = {'Fixed width': '#95a5a6', 'CSD': '#2980b9', 'CiPOT': '#16a085'}
FIG_ALPHA = 0.20


def _hist_panel(ax, pi, title):
    ax.hist(pi, bins=10, range=(0, 1), density=True, color=title[1], alpha=0.85, edgecolor='white')
    ax.axhline(1.0, color='#e74c3c', ls='--', lw=1.1, label='Uniform (ideal)')
    ax.axvline(pi.mean(), color='black', ls=':', lw=1.0, label='Mean $\\pi$')
    ax.set_title(f'{title[0]} (mean $\\pi$ = {pi.mean():.2f})')
    ax.set_xlim(0, 1)


# Figure 4: Cox PH, three methods
with plt.rc_context(RC):
    fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.1), sharey=True)
    for ax, m in zip(axes, METHODS):
        _hist_panel(ax, PI[('Cox PH', FIG_ALPHA, m)], (m, METHOD_COLORS[m]))
        ax.set_xlabel('Within-interval percentile $\\pi$')
    axes[0].set_ylabel('Density')
    axes[0].legend(loc='upper right', frameon=False)
    fig.tight_layout()
    path = PAPER_FIG_DIR / 'WI_three_methods_CoxPH.png'
    fig.savefig(path, dpi=300, bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.show(); plt.close(fig)
    print('saved', path)

# Supplementary Figure S1: all four models (rows) x three methods (columns)
with plt.rc_context(RC):
    fig, axes = plt.subplots(len(MODELS), 3, figsize=(7.0, 1.6 * len(MODELS) + 0.3),
                             sharex=True, sharey=True)
    for r, name in enumerate(MODELS):
        for c, m in enumerate(METHODS):
            _hist_panel(axes[r, c], PI[(name, FIG_ALPHA, m)], (f'{name}: {m}', METHOD_COLORS[m]))
        axes[r, 0].set_ylabel('Density')
    for ax in axes[-1, :]:
        ax.set_xlabel('Within-interval percentile $\\pi$')
    axes[0, 0].legend(loc='upper right', frameon=False)
    fig.tight_layout()
    path = PAPER_FIG_DIR / 'WI_three_methods_all_models.png'
    fig.savefig(path, dpi=300, bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.show(); plt.close(fig)
    print('saved', path)

### 9.3 Figure 5: per-subject intervals
Intervals for every evaluation subject under the three constructions at $\alpha = 0.20$, for Cox PH (Figure 5) and all four models (Supplementary Figure S2).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FIG5_ALPHA = 0.20
R_LO, R_HI = 0.90, 0.10            # interval = [rho 0.90 time, rho 0.10 time] at alpha = 0.20
FIG5_YMAX = 25

BOUNDS = {}   # BOUNDS[(model, method)] = (lo, hi, sort_key, dur, ev)
for name, (s_cal, s_eval, R_fixed, ev_e, dur_e) in MODELS.items():
    q_csd = csd_calibrate(compute_pcts(s_cal, RHO_EXT), dur_cal_all, ev_cal_all, RHO_EXT)
    pcts_csd = enforce_monotonic_pcts(np.clip(csd_adjust_pcts(compute_pcts(s_eval, RHO_EXT), q_csd), 0, None))
    o_cal = ev_cal_all == 1
    ipot = _surv_at(s_cal, dur_cal_all)[o_cal]
    p_levels = np.array([np.quantile(ipot, _finite_level(r, len(ipot))) for r in RHO_EXT])
    pcts_cipot = enforce_monotonic_pcts(compute_pcts(s_eval, p_levels))
    mid = COL[0.50]
    lo_f, hi_f = np.asarray(R_fixed[FIG5_ALPHA]['lo']), np.asarray(R_fixed[FIG5_ALPHA]['hi'])
    BOUNDS[(name, 'Fixed width')] = (lo_f, hi_f, (lo_f + hi_f) / 2, dur_e, ev_e)
    BOUNDS[(name, 'CSD')] = (pcts_csd[:, COL[R_LO]], pcts_csd[:, COL[R_HI]], pcts_csd[:, mid], dur_e, ev_e)
    BOUNDS[(name, 'CiPOT')] = (pcts_cipot[:, COL[R_LO]], pcts_cipot[:, COL[R_HI]], pcts_cipot[:, mid], dur_e, ev_e)


def _interval_panel(ax, lo, hi, key, dur, ev, title):
    lo, hi, key, dur, ev = (np.asarray(a, dtype=float) for a in (lo, hi, key, dur, ev))
    order = np.argsort(key)
    xs = np.arange(len(order))
    lo, hi, dur, ev = lo[order], hi[order], dur[order], ev[order]
    obs = ev == 1
    cov = (dur >= lo) & (dur <= hi)
    ax.vlines(xs[~obs], lo[~obs], hi[~obs], color='lightgray', lw=0.8, label='Censored', zorder=1)
    ci, nc = obs & cov, obs & ~cov
    ax.vlines(xs[ci], lo[ci], hi[ci], color='#2980b9', lw=1.0, zorder=2)
    ax.scatter(xs[ci], dur[ci], color='#2980b9', marker='D', s=9, label='Covered', zorder=3)
    ax.vlines(xs[nc], lo[nc], hi[nc], color='#c0392b', lw=1.0, zorder=2)
    ax.scatter(xs[nc], dur[nc], color='#c0392b', marker='x', s=12, label='Not covered', zorder=3)
    ax.set_title(title)
    ax.set_ylim(0, FIG5_YMAX)
    print(f'{title:<28} coverage {ci.sum()}/{obs.sum()} = {ci.sum() / obs.sum():.1%} | '
          f'missed below {int((obs & (dur < lo)).sum())}, missed above {int((obs & (dur > hi)).sum())}')


def interval_fig_three(names, fname):
    with plt.rc_context(RC):
        fig, axes = plt.subplots(len(names), 3, figsize=(7.0, 1.75 * len(names) + 0.45),
                                 sharex=True, sharey=True, squeeze=False)
        for r, name in enumerate(names):
            for c, m in enumerate(METHODS):
                title = m if len(names) == 1 else f'{name}: {m}'
                _interval_panel(axes[r, c], *BOUNDS[(name, m)], title)
            axes[r, 0].set_ylabel('Years from baseline')
        axes[0, 0].legend(loc='upper left', frameon=False)
        for ax in axes[-1, :]:
            ax.set_xlabel('Subjects (sorted)')
        fig.tight_layout()
        path = PAPER_FIG_DIR / fname
        fig.savefig(path, dpi=300, bbox_inches='tight', pad_inches=0.03, facecolor='white')
        plt.show(); plt.close(fig)
        print('saved', path, '\n')


interval_fig_three(['Cox PH'], 'CI_three_methods_CoxPH.png')            # Figure 5
interval_fig_three(list(MODELS), 'CI_three_methods_all_models.png')     # Supplementary Figure S2

### 9.4 Re-centering and sensitivity analyses (Tables 4 and 5)
- **Re-centered fixed width.** Shifts each predicted median by the median calibration residual before building the symmetric interval, which separates the location of the interval from its shape.
- **IPCW-weighted $\pi$.** Weights each observed evaluation conversion by $1/\hat G(T_i)$, so later converters count more.
- **Grid-maximum medians.** Counts subjects whose curve never falls to 0.5, so the predicted median defaults to the end of the time grid, and repeats the fixed-width analysis without them.
- **Full CiPOT.** Also calibrates on censored subjects, whose scores are spread uniformly on $[0, \hat S_i(c_i)]$.

Results are saved to `outputs/sensitivity_checks.csv` and `outputs/grid_max_check.csv`.

In [ ]:
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter

R_CIPOT = 100           # pseudo-uniform draws per subject for full CiPOT
G_FLOOR = 0.05          # same clipping as Section 8

# Censoring survival G(t), fit on the TRAINING split only (event and censoring flipped)
_kmf_c = KaplanMeierFitter().fit(np.asarray(y_dur_mci_train, float),
                                 event_observed=1 - np.asarray(y_ev_mci_train, int))
def _G(t):
    g = _kmf_c.survival_function_at_times(np.asarray(t, float)).values
    return np.clip(g, G_FLOOR, 1.0)


def _median_from_curve(surv):
    """Predicted median: first grid time with S <= 0.5, else the last grid time."""
    grid = surv.index.values.astype(float)
    vals = surv.values
    out = np.full(vals.shape[1], grid[-1])
    for j in range(vals.shape[1]):
        below = np.where(vals[:, j] <= 0.5)[0]
        if len(below):
            out[j] = grid[below[0]]
    return out, grid[-1]


def _q_level(n, alpha):
    return min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)


def _evaluate(lo, hi, dur, ev):
    lo, hi, dur, ev = (np.asarray(a, float) for a in (lo, hi, dur, ev))
    o = ev == 1
    t, l, h = dur[o], lo[o], hi[o]
    w = h - l
    ok = w > 0
    pi = np.clip((t[ok] - l[ok]) / w[ok], 0, 1)
    wts = 1.0 / _G(t[ok])
    return dict(mean_pi=float(pi.mean()),
                ipcw_mean_pi=float(np.sum(wts * pi) / np.sum(wts)),
                coverage_pct=100 * float(((t >= l) & (t <= h)).mean()),
                miss_below=int((t < l).sum()), miss_above=int((t > h).sum()),
                median_width_yr=float(np.median(hi - lo)))


rows, grid_rows = [], []
for name, (s_cal, s_eval, R_fixed, ev_e, dur_e) in MODELS.items():
    ev_e, dur_e = np.asarray(ev_e, int), np.asarray(dur_e, float)
    o_cal = ev_cal_all == 1

    # ── point predictions (curve-inverted median, same rule for every model)
    t_cal, gmax_c = _median_from_curve(s_cal)
    t_ev, gmax_e = _median_from_curve(s_eval)
    resid = dur_cal_all[o_cal] - t_cal[o_cal]          # actual - predicted
    shift = float(np.median(resid))                    # re-centering offset (years)

    # ── Grid-maximum medians
    at_max = np.isclose(t_ev, gmax_e)
    grid_rows.append(dict(model=name, n_eval=len(t_ev), n_at_grid_max=int(at_max.sum()),
                          n_obs_events_at_grid_max=int((at_max & (ev_e == 1)).sum()),
                          grid_max_yr=round(float(gmax_e), 1),
                          median_shift_yr=round(shift, 2)))

    # ── adaptive percentile times (as in Section 9.2)
    q_csd = csd_calibrate(compute_pcts(s_cal, RHO_EXT), dur_cal_all, ev_cal_all, RHO_EXT)
    pcts_csd = enforce_monotonic_pcts(np.clip(csd_adjust_pcts(compute_pcts(s_eval, RHO_EXT), q_csd), 0, None))
    ipot = _surv_at(s_cal, dur_cal_all)[o_cal]
    p_simple = np.array([np.quantile(ipot, _finite_level(r, len(ipot))) for r in RHO_EXT])
    pcts_cp = enforce_monotonic_pcts(compute_pcts(s_eval, p_simple))

    # full CiPOT: events -> R copies of S_i(T_i); censored -> S_i(c_i) * pseudo-uniform grid
    s_at_obs = _surv_at(s_cal, dur_cal_all)
    u = (np.arange(R_CIPOT) + 0.5) / R_CIPOT
    scores_full = np.concatenate([np.repeat(s_at_obs[o_cal], R_CIPOT),
                                  np.outer(s_at_obs[~o_cal], u).ravel()])
    n_subj = len(dur_cal_all)
    p_full = np.array([np.quantile(scores_full, _finite_level(r, n_subj)) for r in RHO_EXT])
    pcts_full = enforce_monotonic_pcts(compute_pcts(s_eval, p_full))

    for a, (r_lo, r_hi) in ALPHAS.items():
        n_ev = int(o_cal.sum())
        # fixed width from curve-inverted medians
        q_plain = float(np.quantile(np.abs(resid), _q_level(n_ev, a)))
        # re-centered fixed width
        c_cal = t_cal[o_cal] + shift
        q_rc = float(np.quantile(np.abs(dur_cal_all[o_cal] - c_cal), _q_level(n_ev, a)))
        c_ev = t_ev + shift

        bounds = {
            'Fixed width': (R_fixed[a]['lo'], R_fixed[a]['hi']),
            'Fixed width (curve median)': (np.clip(t_ev - q_plain, 0, None), t_ev + q_plain),
            'Fixed width, re-centered': (np.clip(c_ev - q_rc, 0, None), c_ev + q_rc),
            'CSD': (pcts_csd[:, COL[r_lo]], pcts_csd[:, COL[r_hi]]),
            'CiPOT (simplified)': (pcts_cp[:, COL[r_lo]], pcts_cp[:, COL[r_hi]]),
            'CiPOT (full, with censored)': (pcts_full[:, COL[r_lo]], pcts_full[:, COL[r_hi]]),
        }
        for m, (lo, hi) in bounds.items():
            rows.append(dict(model=name, alpha=a, method=m, **_evaluate(lo, hi, dur_e, ev_e)))

        # sensitivity: fixed width without the grid-maximum subjects
        keep = ~at_max
        rows.append(dict(model=name, alpha=a, method='Fixed width, excl. grid-max subjects',
                         **_evaluate(np.asarray(R_fixed[a]['lo'])[keep], np.asarray(R_fixed[a]['hi'])[keep],
                                     dur_e[keep], ev_e[keep])))

chk = pd.DataFrame(rows).round(3)
chk.to_csv(OUT_DIR / 'sensitivity_checks.csv', index=False)
grid_df = pd.DataFrame(grid_rows)
grid_df.to_csv(OUT_DIR / 'grid_max_check.csv', index=False)
pd.set_option('display.width', 220)
print(chk.to_string(index=False))
print('\nGrid-maximum medians on the evaluation set:')
print(grid_df.to_string(index=False))


def _cell(df, model, a, method, key='mean_pi'):
    r = df[(df.model == model) & (df.alpha == a) & (df.method == method)].iloc[0]
    return r


# ── Table 5 rows: π (coverage) and median width, four constructions
print('\nTable 5 rows: pi (coverage) & width, for Fixed | Re-centered | CSD | CiPOT')
order = ['Fixed width', 'Fixed width, re-centered', 'CSD', 'CiPOT (simplified)']
short = {'Cox PH': 'Cox PH', 'AFT (Weibull)': 'AFT', 'GBSA': 'GBSA', 'DeepSurv': 'DeepSurv'}
for name in MODELS:
    for i, a in enumerate(ALPHAS):
        cells = []
        for m in order:
            r = _cell(chk, name, a, m)
            cells += [f'{r.mean_pi:.3f} ({r.coverage_pct:.1f})', f'{r.median_width_yr:.1f}']
        label = short[name] if i == 0 else ''
        print(f'{label:<9} & {a:.2f} & ' + ' & '.join(cells) + r' \\')

# ── LaTeX rows: sensitivity table (α = 0.20)
print('\n=== Sensitivity table rows (alpha = 0.20): IPCW-weighted mean π; full CiPOT ===')
for name in MODELS:
    f = _cell(chk, name, 0.20, 'Fixed width')
    rc = _cell(chk, name, 0.20, 'Fixed width, re-centered')
    c = _cell(chk, name, 0.20, 'CSD')
    p = _cell(chk, name, 0.20, 'CiPOT (simplified)')
    full = _cell(chk, name, 0.20, 'CiPOT (full, with censored)')
    print(f'{short[name]:<9} & {f.ipcw_mean_pi:.3f} & {rc.ipcw_mean_pi:.3f} & {c.ipcw_mean_pi:.3f} & '
          f'{p.ipcw_mean_pi:.3f} & {full.mean_pi:.3f} & {full.ipcw_mean_pi:.3f} & {full.coverage_pct:.1f}' + r' \\')


## 10. Five-fold validation (Section 5.7.1, Table 6)
The comparison of fixed width, CSD, and CiPOT repeated over five folds for all four models, pooling the evaluation conversions, with bootstrap 95% CIs ($\alpha = 0.20$). Each fold refits the imputer and the models on that fold's training subjects. This is the slowest step; results are saved per model to `outputs/kfold_three_methods/`, so a rerun reuses finished models.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUN_INTERVAL_SHAPE_KFOLD = True   # refits 4 models x 5 folds (slow)
MODEL_TYPES = ['Cox PH', 'GBSA', 'AFT', 'DeepSurv']
KFOLD_N_TRIALS = 10
KFOLD_OUT = OUT_DIR / 'kfold_three_methods'
KFOLD_OUT.mkdir(parents=True, exist_ok=True)

def _surv_at_kf(surv_matrix, times_obs):
    """Predicted S_i(T_i) for each subject at its own observed time (step function)."""
    grid = surv_matrix.index.values.astype(float)
    vals = surv_matrix.values
    idx = np.searchsorted(grid, np.asarray(times_obs, dtype=float), side='right') - 1
    out = np.ones(len(times_obs))
    has = idx >= 0
    out[has] = vals[idx[has], np.arange(len(times_obs))[has]]
    return out


def _pi(lo, hi, dur, ev):
    o = ev == 1
    w = hi[o] - lo[o]
    v = w > 0
    return np.clip((dur[o][v] - lo[o][v]) / w[v], 0, 1)


def compute_three_pi_arrays(surv_cal, surv_eval, dur_cal, ev_cal, dur_eval, ev_eval, alpha=0.20):
    """Per-subject pi for fixed width, CSD, and CiPOT at alpha = 0.20.
    Uses percentile_levels (0.1 to 0.9) from Section 9.1: interval = [rho 0.9, rho 0.1]."""
    dur_cal, dur_eval = np.asarray(dur_cal, float), np.asarray(dur_eval, float)
    ev_cal, ev_eval = np.asarray(ev_cal, int), np.asarray(ev_eval, int)

    # Fixed width
    t_hat_cal = _invert_surv(surv_cal)
    t_hat_eval = _invert_surv(surv_eval)
    obs_cal = ev_cal == 1
    scores = t_hat_cal[obs_cal] - dur_cal[obs_cal]
    q_hat = _conformal_quantile(scores, alpha, obs_cal.sum())
    pi_fw = _pi(np.clip(t_hat_eval - q_hat, 0, None), t_hat_eval + q_hat, dur_eval, ev_eval)

    # CSD (unchanged)
    pcts_cal = compute_pcts(surv_cal, percentile_levels)
    pcts_eval = compute_pcts(surv_eval, percentile_levels)
    q_hats_csd = csd_calibrate(pcts_cal, dur_cal, ev_cal, percentile_levels)
    pcts_adj = enforce_monotonic_pcts(np.clip(csd_adjust_pcts(pcts_eval, q_hats_csd), 0, None))
    pi_csd = _pi(pcts_adj[:, -1], pcts_adj[:, 0], dur_eval, ev_eval)

    # CiPOT (simplified, observed events only): same grid, recalibrated probability levels
    ipot = _surv_at_kf(surv_cal, dur_cal)[obs_cal]
    n = len(ipot)
    p_levels = np.array([np.quantile(ipot, min(np.ceil((n + 1) * r) / n, 1.0)) for r in percentile_levels])
    pcts_cp = enforce_monotonic_pcts(compute_pcts(surv_eval, p_levels))
    pi_cp = _pi(pcts_cp[:, -1], pcts_cp[:, 0], dur_eval, ev_eval)

    return pi_fw, pi_csd, pi_cp


def fit_model_fold(model_type, X_train, ev_train, dur_train):
    if model_type == 'Cox PH':
        model, scaler = run_cox_ph(X_train, ev_train, dur_train,
                                   label=f'{model_type}Fold', n_trials=KFOLD_N_TRIALS)[2:4]
        return 'cox', model, scaler
    if model_type == 'GBSA':
        model = gbsa_survival_cv(X_train, ev_train, dur_train, feature_names=list(X_train.columns),
                                 label=f'{model_type}Fold', n_trials=KFOLD_N_TRIALS)[3]
        return 'gbsa', model, None
    if model_type == 'AFT':
        return 'aft', _fit_aft_with_cv(X_train, ev_train, dur_train, f'{model_type}Fold'), None
    if model_type == 'DeepSurv':
        model, scaler = run_deepsurv(X_train, ev_train, dur_train,
                                     f'{model_type}Fold', n_trials=KFOLD_N_TRIALS)[2:4]
        return 'deepsurv', model, scaler
    raise ValueError(model_type)


def predict_surv_fold(kind, model, scaler, X, ev, dur):
    if kind == 'cox':
        return calc_cox_ph_c(model, scaler, X, ev, dur)[1]
    if kind == 'gbsa':
        return calc_gbsa_c(model, X, ev, dur)[1]
    if kind == 'aft':
        d = X.copy()
        d['duration'] = np.array(dur, dtype=np.float64)
        d['event'] = np.array(ev, dtype=np.int32)
        return model.predict_survival_function(d)
    if kind == 'deepsurv':
        return calc_deepsurv_c(model, scaler, X, ev, dur)[1]
    raise ValueError(kind)


if RUN_INTERVAL_SHAPE_KFOLD:
    from sklearn.model_selection import StratifiedKFold, train_test_split as _is_tts

    unique_rids_full = surv_mci_labels.index.values
    fold_events_full = surv_mci_labels['event'].values
    skf_is = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    kfold_bootstrap_results = {}

    for model_type in MODEL_TYPES:
        cache = KFOLD_OUT / f'{model_type.replace(" ", "_")}.npz'
        if cache.exists():   # reuse a finished model from a previous run
            z = np.load(cache)
            pooled = {'fixed': z['fixed'], 'csd': z['csd'], 'cipot': z['cipot']}
            print(f'{model_type}: loaded from {cache}')
        else:
            feature_cols = X_mci_train_cox.columns if model_type == 'Cox PH' else X_mci_train.columns
            pooled = {'fixed': [], 'csd': [], 'cipot': []}
            for fold_idx, (trainval_idx, eval_idx) in enumerate(skf_is.split(unique_rids_full, fold_events_full)):
                trainval_rids = unique_rids_full[trainval_idx]
                eval_rids = unique_rids_full[eval_idx]
                train_rids, cal_rids = _is_tts(
                    trainval_rids, test_size=0.20, random_state=RANDOM_SEED + fold_idx,
                    stratify=surv_mci_labels.loc[trainval_rids, 'event'])

                _raw = df_bl.set_index('RID')
                # MICE fit on this fold's training rows only
                X_tr, (X_ca, X_ev) = mice_fit_transform(_raw.loc[train_rids, feature_cols],
                                                        [_raw.loc[cal_rids, feature_cols],
                                                         _raw.loc[eval_rids, feature_cols]])
                lab = lambda rids, c: surv_mci_labels.loc[rids, c].values

                kind, model, scaler = fit_model_fold(model_type, X_tr, lab(train_rids, 'event'), lab(train_rids, 'duration'))
                s_ca = predict_surv_fold(kind, model, scaler, X_ca, lab(cal_rids, 'event'), lab(cal_rids, 'duration'))
                s_ev = predict_surv_fold(kind, model, scaler, X_ev, lab(eval_rids, 'event'), lab(eval_rids, 'duration'))

                fw, csd, cp = compute_three_pi_arrays(s_ca, s_ev, lab(cal_rids, 'duration'), lab(cal_rids, 'event'),
                                                      lab(eval_rids, 'duration'), lab(eval_rids, 'event'))
                pooled['fixed'].extend(fw); pooled['csd'].extend(csd); pooled['cipot'].extend(cp)
                print(f'{model_type}: fold {fold_idx + 1}/5 done (n fixed={len(fw)}, csd={len(csd)}, cipot={len(cp)})')
            pooled = {k: np.asarray(v) for k, v in pooled.items()}
            np.savez(cache, **pooled)
            print(f'{model_type}: saved {cache}')

        res = {k: (*bootstrap_ci(list(v)), len(v)) for k, v in pooled.items()}  # (mean, lo, hi, n)
        kfold_bootstrap_results[model_type] = res
        for k in ['fixed', 'csd', 'cipot']:
            m, lo, hi, n = res[k]
            print(f'   {k:<6} mean_pi={m:.3f} 95%CI=[{lo:.3f}, {hi:.3f}] (n={n})')
        for k in ['csd', 'cipot']:
            overlap = not (res['fixed'][2] < res[k][1] or res[k][2] < res['fixed'][1])
            print(f'   fixed vs {k}: CIs overlap = {overlap}')
        print()

    # ── Save, and print the Table 6 rows ─────────────────────────────────────
    tbl = pd.DataFrame({m: {k: '%.3f [%.3f, %.3f]' % kfold_bootstrap_results[m][k][:3]
                            for k in ['fixed', 'csd', 'cipot']} for m in MODEL_TYPES}).T
    tbl.to_csv(OUT_DIR / 'table6_kfold_three_methods.csv')
    with open(OUT_DIR / 'table6_kfold_three_methods.json', 'w') as f:
        json.dump({m: {k: list(map(float, v[:3])) + [int(v[3])] for k, v in r.items()}
                   for m, r in kfold_bootstrap_results.items()}, f, indent=2)
    print('Table 6 rows:')
    for m in ['Cox PH', 'AFT', 'GBSA', 'DeepSurv']:
        r = kfold_bootstrap_results[m]
        cells = [r'\makecell{%.3f\\ {\scriptsize [%.3f, %.3f]}}' % r[k][:3] for k in ['fixed', 'csd', 'cipot']]
        print(f'{m} & ' + ' & '.join(cells) + r' \\')
    print('pooled observed events per model:', {m: kfold_bootstrap_results[m]['fixed'][3] for m in MODEL_TYPES})
else:
    print('Skipped (RUN_INTERVAL_SHAPE_KFOLD = False).')

## 11. Feature importance (Section 5.7.2)
SHAP values for each model on calibration-set subjects, aggregated into six feature families as a share of each model's total SHAP mass. The Cox, AFT, and DeepSurv explainers sample randomly, so the random seed is reset before each.

### 11.1 Cox PH

In [ ]:
import shap
np.random.seed(RANDOM_SEED)

def cox_model_predict(X):
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=X_mci_train_cox.columns)
    X_scaled = pd.DataFrame(cox_scaler_mci.transform(X.values), columns=X.columns)
    return cox_model_mci.predict_partial_hazard(X_scaled).values

cox_explainer = shap.KernelExplainer(
    cox_model_predict, X_mci_train_cox.sample(100, random_state=RANDOM_SEED))
sample_size_cox = min(100, len(X_mci_cal_cox))
X_cal_sample_cox = X_mci_cal_cox.sample(sample_size_cox, random_state=RANDOM_SEED)
shap_values_cox = cox_explainer.shap_values(X_cal_sample_cox)

### 11.2 GBSA

In [ ]:
np.random.seed(RANDOM_SEED)
try:
    gbsa_explainer = shap.TreeExplainer(gbsa_model_mci)
    shap_values_gbsa = gbsa_explainer.shap_values(X_mci_cal)
    X_gbsa_shap_input = X_mci_cal
except Exception as e:
    print(f'TreeExplainer failed ({e}): falling back to KernelExplainer')
    def gbsa_model_predict(X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=X_mci_train.columns)
        return gbsa_model_mci.predict(X)
    gbsa_explainer = shap.KernelExplainer(
        gbsa_model_predict, X_mci_train.sample(100, random_state=RANDOM_SEED))
    X_gbsa_shap_input = X_mci_cal.sample(min(100, len(X_mci_cal)), random_state=RANDOM_SEED)
    shap_values_gbsa = gbsa_explainer.shap_values(X_gbsa_shap_input)

### 11.3 AFT

In [ ]:
import shap
np.random.seed(RANDOM_SEED)

def model_predict(X):
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=X_mci_train.columns)
    return aft_model_mci.predict_expectation(X).values

explainer = shap.KernelExplainer(model_predict,
                                  X_mci_train.sample(100, random_state=RANDOM_SEED))
sample_size = min(100, len(X_mci_test))
X_test_sample_raw = X_mci_cal.sample(sample_size, random_state=RANDOM_SEED)
shap_values = explainer.shap_values(X_test_sample_raw)

### 11.4 DeepSurv

In [ ]:
np.random.seed(RANDOM_SEED)
def ds_model_predict(X):
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=X_mci_train.columns)
    X_scaled = ds_scaler_mci.transform(X.values).astype('float32')
    return ds_model_mci.predict(X_scaled).flatten()

ds_explainer = shap.KernelExplainer(
    ds_model_predict, X_mci_train.sample(100, random_state=RANDOM_SEED))
sample_size_ds = min(100, len(X_mci_cal))
X_cal_sample_ds = X_mci_cal.sample(sample_size_ds, random_state=RANDOM_SEED)
shap_values_ds = ds_explainer.shap_values(X_cal_sample_ds)

### 11.5 Rankings compared across models
Spearman correlation between the models' SHAP-based feature rankings, on the features all four models share.

In [ ]:
from scipy.stats import spearmanr

common_features = [c for c in X_mci_train_cox.columns if c in X_mci_train.columns]

def mean_abs_shap(shap_vals, feature_names, common):
    s = pd.Series(np.abs(shap_vals).mean(axis=0), index=feature_names)
    return s.reindex(common)

shap_values_aft = shap_values

imp_shap_aft  = mean_abs_shap(shap_values_aft,  X_test_sample_raw.columns, common_features)
imp_shap_cox  = mean_abs_shap(shap_values_cox,  X_cal_sample_cox.columns,  common_features)
imp_shap_gbsa = mean_abs_shap(shap_values_gbsa, X_gbsa_shap_input.columns, common_features)
imp_shap_ds   = mean_abs_shap(shap_values_ds,   X_cal_sample_ds.columns,   common_features)

shap_comparison = pd.DataFrame({
    'AFT': imp_shap_aft,
    'Cox PH': imp_shap_cox,
    'GBSA': imp_shap_gbsa,
    'DeepSurv': imp_shap_ds,
})

# Rank within each model (1 = most important)
shap_rank = shap_comparison.rank(ascending=False, method='min')

print('Mean |SHAP value| by feature and model (common feature set):')
print(shap_comparison.round(4).to_string())
print('\nRank (1 = most important) by feature and model:')
print(shap_rank.astype(int).to_string())

# Spearman rank correlation between models' SHAP-based feature rankings
model_names = shap_rank.columns.tolist()
corr_matrix = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
for m1 in model_names:
    for m2 in model_names:
        corr_matrix.loc[m1, m2] = spearmanr(shap_rank[m1], shap_rank[m2]).correlation

print('\nSpearman rank correlation between models (SHAP-based feature rankings):')
print(corr_matrix.round(3).to_string())

### 11.6 Feature families
Share of each model's total SHAP mass by feature family (Figure 6 is saved in Section 12).

In [ ]:
FEATURE_FAMILIES = {
    'Demographics': ['AGE', 'PTGENDER_num', 'PTEDUCAT'],
    'Genetic / APOE': ['APOE4'],
    'APOE4 Interactions': ['APOE4_x_amyloid', 'APOE4_x_tau',
                            'APOE4_x_hippo_ICV', 'APOE4_amyloid_pos'],
    'Cognitive Assessments': ['MMSE', 'CDRSB', 'ADAS11', 'ADAS13', 'LDELTOTAL',
                               'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting',
                               'FAQ', 'MOCA', 'EcogPtTotal', 'EcogSPTotal',
                               'mPACCdigit', 'mPACCtrailsB'],
    'MRI Volumetric (Imaging)': ['Hippocampus_ICV', 'Entorhinal_ICV', 'Ventricles_ICV',
                                  'Fusiform_ICV', 'MidTemp_ICV', 'WholeBrain_ICV',
                                  'hippo_entorhinal_ratio', 'midtemp_fusiform_ratio'],
    'CSF/PET Biomarkers': ['ABETA', 'TAU', 'PTAU', 'FDG', 'AV45'],
}

def feature_to_family(feat):
    for family, feats in FEATURE_FAMILIES.items():
        if feat in feats:
            return family
    return 'Other'

def family_shap_totals(shap_vals, feature_names):
    """Sum of mean(|SHAP|) per family, using this model's own full feature set."""
    per_feature = pd.Series(np.abs(shap_vals).mean(axis=0), index=feature_names)
    per_feature_df = per_feature.to_frame('mean_abs_shap')
    per_feature_df['family'] = [feature_to_family(f) for f in per_feature_df.index]
    return per_feature_df.groupby('family')['mean_abs_shap'].sum()

family_totals = pd.DataFrame({
    'AFT':      family_shap_totals(shap_values_aft,  X_test_sample_raw.columns),
    'Cox PH':   family_shap_totals(shap_values_cox,  X_cal_sample_cox.columns),
    'GBSA':     family_shap_totals(shap_values_gbsa, X_gbsa_shap_input.columns),
    'DeepSurv': family_shap_totals(shap_values_ds,   X_cal_sample_ds.columns),
})
family_totals = family_totals.reindex(FEATURE_FAMILIES.keys()).fillna(0)

print('Sum of mean(|SHAP value|) by feature family and model:')
print(family_totals.round(4).to_string())

# Normalize to % of each model's total SHAP mass (raw magnitudes are not comparable across models)
family_pct = family_totals.div(family_totals.sum(axis=0), axis=1) * 100
print('\nAs % of each model\'s total SHAP mass:')
print(family_pct.round(1).to_string())

# ── Rank of families within each model ────────────────────────────────────────
family_rank = family_pct.rank(ascending=False, method='min')
print('\nFamily rank within each model (1 = most important):')
print(family_rank.astype(int).to_string())

## 12. Paper numbers and figures
Collects every value reported in the paper into `outputs/paper_numbers.json`, saves Figures 2, 3, and 6 at 300 dpi to `figures/paper/` (Figures 4, 5, S1, and S2 are saved in Section 9), and zips the outputs.

In [ ]:
# ── Every value reported in the paper, saved to outputs/paper_numbers.json ────
import json

paper_numbers = {}

def _events(ev):
    return np.asarray(ev) == 1

def _section(title):
    print('\n' + '=' * 78 + '\n' + title + '\n' + '=' * 78)

def _csv(name):
    p = OUT_DIR / name
    return pd.read_csv(p) if p.exists() else None

# Cohort and split (Sections 4.1 and 5.1)
_section('Cohort and split')
d = dict(n_cohort=int(len(surv_mci_labels)), n_events=int(surv_mci_labels['event'].sum()),
         event_rate_pct=round(float(surv_mci_labels['event'].mean() * 100), 1),
         n_features=int(X_mci_train.shape[1]), n_features_cox=int(X_mci_train_cox.shape[1]))
for k, (X, ev) in {'train': (X_mci_train, y_ev_mci_train), 'cal': (X_mci_cal, y_ev_mci_cal),
                   'eval': (X_mci_eval, y_ev_mci_eval)}.items():
    d[f'n_{k}'], d[f'events_{k}'] = int(len(X)), int(_events(ev).sum())
print(json.dumps(d, indent=2))
paper_numbers['cohort_and_split'] = d

# Table 2: C-td (Section 5.2)
_section('Table 2: C-td on the calibration set')
print(results_df.to_string(index=False))
paper_numbers['table2_ctd'] = results_df.to_dict('records')
boot = _csv('model_comparison_pairwise_ci.csv')
if boot is not None:
    print('\nPairwise bootstrap (C-td difference, 95% CI):')
    print(boot.round(4).to_string(index=False))
    paper_numbers['pairwise_bootstrap'] = boot.round(4).to_dict('records')

# Agreement on predicted timing (Section 5.2)
_section('Agreement on predicted median conversion time')
print(pair_df.round(3).to_string(index=False))
paper_numbers['time_consistency'] = pair_df.round(3).to_dict('records')

# Fixed-width intervals (Section 5.3)
_section('Fixed-width intervals: mean pi and coverage')
out = {}
for name, (R, ev, dur) in {'AFT': (results_cp_aft, ev_eval_aft, dur_eval_aft),
                           'Cox PH': (results_cp_cox, ev_eval_cox, dur_eval_cox),
                           'GBSA': (results_cp_gbsa, ev_eval_gbsa, dur_eval_gbsa),
                           'DeepSurv': (results_cp_deepsurv, ev_eval_ds, dur_eval_ds)}.items():
    out[name] = {}
    for a in (0.10, 0.20):
        r, o = R[a], _events(ev)
        lo, hi, t = np.asarray(r['lo'])[o], np.asarray(r['hi'])[o], np.asarray(dur)[o]
        ok = (hi - lo) > 0
        tag = int(round(a * 100))
        out[name][f'mean_pi_a{tag}'] = round(float(np.clip((t[ok] - lo[ok]) / (hi - lo)[ok], 0, 1).mean()), 3)
        out[name][f'coverage_a{tag}'] = round(float(r['coverage']), 4)
        out[name][f'q_hat_a{tag}_yr'] = round(float(r['q_hat']), 3)
print(pd.DataFrame(out).T.to_string())
paper_numbers['fixed_width'] = out
gm = _csv('grid_max_check.csv')
if gm is not None:
    print('\nSubjects whose predicted median is the end of the time grid:')
    print(gm.to_string(index=False))
    paper_numbers['grid_max'] = gm.to_dict('records')

# Landmark analysis (Section 5.4, Table 3)
_section('Landmark analysis')
d = {'single_split_mean_pi_by_tau': {str(t): round(float(v['mean_pct']), 3)
                                     for t, v in sorted(lm_results_by_landmark.items())}}
if 'visit_sensitivity_df' in globals():
    d['visit_sensitivity'] = visit_sensitivity_df.round(3).to_dict('records')
lmk = _csv('landmark_pi_by_tau.csv')
if lmk is not None:
    print(lmk.round(3).to_string(index=False))
    d['table3_kfold'] = lmk.round(3).to_dict('records')
print(json.dumps({k: v for k, v in d.items() if k != 'table3_kfold'}, indent=2))
paper_numbers['landmark'] = d

# Censoring (Section 5.5, Table 4)
_section('Censoring')
d = dict(weight_min=round(float(ipcw_weights.min()), 2), weight_max=round(float(ipcw_weights.max()), 2),
         weight_mean=round(float(ipcw_weights.mean()), 2),
         censoring_features_significant_uncorrected=int(n_significant_uncorrected),
         censoring_features_significant_bh=int(n_significant_bh),
         reweighted_calibration_mean_pi_a20=ipcw_calibration_df.round(3).to_dict('index'))
print(json.dumps(d, indent=2))
paper_numbers['censoring'] = d

# Adaptive intervals (Section 5.6, Tables 4 and 5)
rf, rc = _csv('three_methods_summary.csv'), _csv('sensitivity_checks.csv')
if rf is not None:
    _section('Fixed width vs CSD vs CiPOT: misses pooled across models')
    print(rf.groupby(['alpha', 'method'])[['miss_below', 'miss_above']].sum().to_string())
    paper_numbers['three_methods'] = rf.to_dict('records')
if rc is not None:
    _section('Table 5 and sensitivity analyses: ranges across models at alpha = 0.20')
    a20 = rc[rc.alpha == 0.20]
    for m in a20['method'].unique():
        v = a20[a20.method == m]
        print(f'{m:<38} mean pi {v.mean_pi.min():.3f}-{v.mean_pi.max():.3f} | '
              f'IPCW-weighted pi {v.ipcw_mean_pi.min():.3f}-{v.ipcw_mean_pi.max():.3f} | '
              f'median width {v.median_width_yr.min():.2f}-{v.median_width_yr.max():.2f} yr')
    paper_numbers['sensitivity_checks'] = rc.to_dict('records')

# Five-fold validation (Section 5.7.1, Table 6)
if 'kfold_bootstrap_results' in globals():
    _section('Table 6: five-fold pooled mean pi with bootstrap 95% CIs')
    d = {m: {k: [round(float(x), 3) for x in v[k][:3]] + [int(v[k][3])] for k in v}
         for m, v in kfold_bootstrap_results.items()}
    for m, v in d.items():
        print(f'{m:<9} ' + '   '.join(f'{k} {x[0]:.3f} [{x[1]:.3f}, {x[2]:.3f}]' for k, x in v.items()))
    paper_numbers['table6_kfold'] = d

# Feature importance (Section 5.7.2)
_section('Feature importance')
print("Share of each model's SHAP mass by family (%):")
print(family_pct.round(1).to_string())
print('\nSpearman correlation of SHAP feature rankings:')
print(corr_matrix.round(3).to_string())
named = ['AV45', 'CDRSB', 'LDELTOTAL', 'mPACCtrailsB', 'APOE4']
print('\nRanks of features named in the text (1 = most important):')
print(shap_rank.reindex(named).to_string())
paper_numbers['feature_importance'] = dict(family_pct=family_pct.round(1).to_dict(),
                                           spearman=corr_matrix.round(3).to_dict(),
                                           named_ranks=shap_rank.reindex(named).to_dict())

def _default(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

with open(OUT_DIR / 'paper_numbers.json', 'w') as f:
    json.dump(paper_numbers, f, indent=2, default=_default)
print('\nSaved', OUT_DIR / 'paper_numbers.json')

In [ ]:
# ── Figures 2, 3, and 6 at 300 dpi ────────────────────────────────────────────
from lifelines import KaplanMeierFitter

PAPER_DPI = 300
PAPER_FIG_DIR = FIG_DIR / 'paper'
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_COLORS = {'AFT (Weibull)': '#3498db', 'AFT': '#3498db', 'Cox PH': '#e74c3c',
                'GBSA': '#2ecc71', 'DeepSurv': '#9b59b6'}
RC = {'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7.5, 'legend.fontsize': 6.5,
      'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'axes.linewidth': 0.7}

def _save(fig, name):
    path = PAPER_FIG_DIR / name
    fig.savefig(path, dpi=PAPER_DPI, bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.show()
    plt.close(fig)
    print('saved', path)

# Figure 2: Kaplan-Meier curve of the MCI-to-AD cohort
with plt.rc_context(RC):
    fig, ax = plt.subplots(figsize=(3.4, 2.4))
    kmf = KaplanMeierFitter()
    kmf.fit(surv_mci_labels['duration'], event_observed=surv_mci_labels['event'], label='MCI to AD')
    kmf.plot_survival_function(ax=ax, color='#c0392b', ci_show=True, ci_alpha=0.15, lw=1.2)
    ax.axhline(0.5, color='gray', ls=':', lw=0.8)
    ax.set(xlabel='Years from baseline', ylabel='Survival probability')
    ax.legend(loc='upper right', frameon=False)
    _save(fig, 'KM_overall.png')

# Figure 3: predicted survival curves from all four models for one calibration-set subject
i = 56
curves = [('GBSA', surv_gbsa_mci.iloc[:, i]), ('AFT (Weibull)', aft_surv_mci.iloc[:, i]),
          ('DeepSurv', surv_mci.iloc[:, i]), ('Cox PH', surv_cox_mci.iloc[:, i])]
dur = float(y_dur_mci_test.iloc[i] if hasattr(y_dur_mci_test, 'iloc') else y_dur_mci_test[i])
with plt.rc_context(RC):
    fig, ax = plt.subplots(figsize=(5.2, 2.5))
    for name, s in curves:
        t, v = np.asarray(s.index, dtype=float), np.asarray(s.values, dtype=float)
        s_at = v[min(np.searchsorted(t, dur), len(v) - 1)]
        ax.step(t, v, where='post', color=MODEL_COLORS[name], lw=1.3, label=f'{name}, S({dur:.2f}) = {s_at:.3f}')
        ax.plot([dur], [s_at], 'D', color=MODEL_COLORS[name], ms=4, mec='black', mew=0.5, zorder=5)
    ax.axvline(dur, color='gray', ls='--', lw=0.8)
    ax.set(xlabel='Time (years)', ylabel='Survival probability S(t)', ylim=(0, 1.03))
    ax.legend(loc='upper right', frameon=False)
    _save(fig, 'All_model_survival_curves.png')

# Figure 6: SHAP importance by feature family
with plt.rc_context(RC):
    fig, ax = plt.subplots(figsize=(6.0, 2.7))
    x = np.arange(len(family_pct.index))
    for k, model in enumerate(family_pct.columns):
        ax.bar(x + (k - 1.5) * 0.2, family_pct[model], 0.2, label=model, color=MODEL_COLORS.get(model))
    ax.set_xticks(x)
    ax.set_xticklabels(family_pct.index, rotation=15, ha='right')
    ax.set_ylabel("% of model's total SHAP mass")
    ax.legend(frameon=False)
    fig.tight_layout()
    _save(fig, 'feat_fam_import.png')

In [ ]:
# Zip figures and outputs for download
import zipfile

zip_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path(OUT_DIR).parent
zip_path = zip_dir / 'paper_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in (FIG_DIR, OUT_DIR):
        for p in Path(folder).rglob('*'):
            if p.is_file():
                z.write(p, p.relative_to(Path(folder).parent))
print('Wrote', zip_path, f'({zip_path.stat().st_size / 1e6:.1f} MB)')
print('Final figures are in', PAPER_FIG_DIR if 'PAPER_FIG_DIR' in globals() else FIG_DIR)